# Grounded LLM explanations for healthy food recommendations

The full pipeline behind the master's thesis: data cleaning, the recommender, explanation generation, grounding verification with two LLM judges, and validation of those judges. See the README for setup, the data needed and how each section maps to the thesis.

The generation and judging cells call an OpenAI-compatible gateway (Lehmus AI) and need API keys in a `.env` file. All other cells run locally.

## 1. Data Loading and Exploration
Load the preprocessed HUMMUS recipes and inspect structure, key columns and missing values.

In [ ]:
import ast
import difflib
import json
import os
import re
import shutil
import time
from collections import Counter

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation
from sklearn.metrics import cohen_kappa_score

load_dotenv()   # API keys are read from .env and never written in the notebook

In [ ]:
import pandas as pd

# Load the preprocessed recipes
recipes = pd.read_csv('data/pp_recipes.csv', low_memory=False)

# Show the shape - how many rows and columns
print(f"Recipes: {recipes.shape[0]} rows, {recipes.shape[1]} columns")

# Show all column names
print("\nAll columns:")
for col in recipes.columns:
    print(f"  {col}")

In [ ]:
# Show the first 3 recipes - see what the data actually looks like
recipes.head(3)

In [ ]:
# Check the columns we care most about
key_columns = [
    'title',
    'tags',
    'ingredients',
    'ingredient_food_kg_names',  
    'calories [cal]',
    'protein [g]',
    'sodium [mg]',
    'totalFat [g]',
    'saturatedFat [g]',
    'sugars [g]',
    'dietaryFiber [g]',
    'who_score',
    'fsa_score',
    'nutri_score',
    'average_rating',
    'number_of_ratings',
    'duration',
    'serves',
    'recipe_url',            # link to full recipe for participants
    'normalization_comment', # data-quality flag - drives the clean-only filter
    'servingSize [g]',       
    'servingsPerRecipe',    
]


print("Columns we need - present or missing:")
for col in key_columns:
    status = "✓ present" if col in recipes.columns else "✗ MISSING"
    print(f"  {status}: {col}")

In [ ]:
print("Missing values in key columns:")
for col in key_columns:
    if col in recipes.columns:
        missing = recipes[col].isna().sum()
        pct = (missing / len(recipes)) * 100
        print(f"  {col}: {missing:,} missing ({pct:.1f}%)")

In [ ]:
print("WHO Score distribution:")
print(recipes['who_score'].describe())
print("\nFSA Score distribution:")
print(recipes['fsa_score'].describe())
print("\nNutriScore distribution:")
print(recipes['nutri_score'].value_counts())

In [ ]:
print("Example tags from first 5 recipes:")
for i, row in recipes.head(5).iterrows():
    print(f"\nRecipe: {row['title']}")
    print(f"Tags: {row['tags']}")

In [ ]:
print("Example ingredients from first 3 recipes:")
for i, row in recipes.head(3).iterrows():
    print(f"\nRecipe: {row['title']}")
    print(f"Ingredients: {row['ingredients']}")

## 2. Data Cleaning
Filter to reliably normalised recipes (`normalization_comment` empty), convert the Nutri-Score to A–E letters, extract and clean ingredient names, and save the cleaned dataset.

In [ ]:
# Clean-only filter, grounded in the HUMMUS scoring implementation.
# The dataset's score_util computes WHO/FSA/NutriScore ONLY for recipes where
# normalization_comment == '' (empty). Recipes marked '*' or '**' had faulty
# serving-size data and their scores are unreliable, so we exclude them
clean_recipes = recipes[
    recipes['who_score'].notna() &
    (recipes['number_of_ratings'] >= 1) &
    (recipes['normalization_comment'].isna() | (recipes['normalization_comment'] == ''))
].copy()

print(f"Full dataset:    {len(recipes):,} recipes")
print(f"Clean dataset:   {len(clean_recipes):,} recipes")
print(f"Removed:         {len(recipes) - len(clean_recipes):,} recipes")

print("\nNormalization flags remaining in clean set (should be empty/NaN only):")
print(clean_recipes['normalization_comment'].value_counts(dropna=False))

In [ ]:
# Convert normalised nutri_score back to A-E letters
nutri_map = {1.00: 'A', 0.75: 'B', 0.50: 'C', 0.25: 'D', 0.00: 'E'}
clean_recipes['nutri_grade'] = clean_recipes['nutri_score'].map(nutri_map)

print("NutriScore grade distribution in clean dataset:")
print(clean_recipes['nutri_grade'].value_counts())

In [ ]:
def extract_ingredient_names(ingredients_raw):
    """
    Extract just the ingredient names from the complex format.
    Returns a list of lowercase ingredient name strings.
    """
    try:
        # The column holds a dict, or a string that has to be parsed first
        if isinstance(ingredients_raw, str):
            data = ast.literal_eval(ingredients_raw)
        else:
            data = ingredients_raw
        
        names = []
        for category, items in data.items():
            for item in items:
                # Each item is a tuple and the first element is the name
                if isinstance(item, tuple) and len(item) > 0:
                    names.append(item[0].lower().strip())
        return names
    except:
        return []


for i, row in clean_recipes.head(3).iterrows():
    names = extract_ingredient_names(row['ingredients'])
    print(f"\n{row['title']}")
    print(f"Ingredients: {names}")

In [ ]:
print("Extracting ingredient names... (this takes a moment)")
clean_recipes['ingredient_names'] = clean_recipes['ingredients'].apply(extract_ingredient_names)
print("Done.")

print(f"\nSample:")
print(clean_recipes[['title', 'ingredient_names']].head(3).to_string())

In [ ]:
def clean_ingredient_for_display(name):
    """Tidy a parsed ingredient name for display to participants.
    Strips parenthetical notes, quantity fragments, and leading 'of'.
    Filtering still uses the raw ingredient_names; this is display-only."""
    n = name.lower().strip()
    n = re.sub(r'\([^)]*\)', '', n)        # remove (...) notes
    n = n.split(',')[0]                     # take part before first comma
    n = re.sub(r'^of\s+', '', n)            # 'of lamb' -> 'lamb'
    n = re.sub(r'\s+', ' ', n).strip()      # collapse whitespace
    return n

def display_ingredients(ingredient_names, max_items=12):
    """Return a clean, de-duplicated ingredient list for display."""
    if not isinstance(ingredient_names, list):
        return []
    seen, out = set(), []
    for raw in ingredient_names:
        c = clean_ingredient_for_display(raw)
        if c and c not in seen:
            seen.add(c)
            out.append(c)
    return out[:max_items]

# Shown on the recipe cards and used to pick the key ingredients
clean_recipes['ingredients_display'] = clean_recipes['ingredient_names'].apply(display_ingredients)

# Check it
for _, row in clean_recipes.head(3).iterrows():
    print(f"{row['title']}")
    print(f"  raw:     {row['ingredient_names'][:4]}")
    print(f"  display: {row['ingredients_display']}\n")

In [ ]:
clean_recipes.to_csv('data/clean_recipes.csv', index=False)
print(f"Saved {len(clean_recipes):,} clean recipes to data/clean_recipes.csv")

## 3. Tag Analysis and Matching Configuration
Parse tags and define the dietary, cuisine and health-goal mappings used for filtering and scoring. All thresholds are grounded in the HUMMUS README (WHO and FSA tables) or documented as design decisions.

In [ ]:
tagged = clean_recipes[clean_recipes['tags'].notna()].copy()
print(f"Recipes with tags: {len(tagged):,} out of {len(clean_recipes):,}")
print(f"Percentage with tags: {len(tagged)/len(clean_recipes)*100:.1f}%")

In [ ]:
def parse_tags(tags_raw):
    """Parse tags from string representation to list."""
    try:
        if isinstance(tags_raw, list):
            return tags_raw
        if isinstance(tags_raw, str) and tags_raw.startswith('['):
            return ast.literal_eval(tags_raw)
        return []
    except:
        return []

# Get all unique tags across the dataset
all_tags = set()
for tags_raw in tagged['tags']:
    tag_list = parse_tags(tags_raw)
    all_tags.update(tag_list)

print(f"Total unique tags: {len(all_tags)}")
print("\nAll tags (sorted):")
for tag in sorted(all_tags):
    print(f"  {tag}")

In [ ]:
# Search for tags related to dietary restrictions and cuisine
dietary_keywords = ['vegetarian', 'vegan', 'gluten', 'dairy', 'low-sodium', 
                    'low-fat', 'low-calorie', 'low-sugar', 'low-carb',
                    'high-protein', 'diabetic', 'halal', 'kosher']

cuisine_keywords = ['italian', 'asian', 'mediterranean', 'mexican', 'greek',
                    'chinese', 'indian', 'french', 'japanese', 'thai',
                    'middle-eastern', 'american', 'european', 'african']

print("DIETARY TAGS FOUND:")
for tag in sorted(all_tags):
    for kw in dietary_keywords:
        if kw in tag.lower():
            print(f"  {tag}")
            break

print("\nCUISINE TAGS FOUND:")
for tag in sorted(all_tags):
    for kw in cuisine_keywords:
        if kw in tag.lower():
            print(f"  {tag}")
            break

### Tag mappings used for matching

In [ ]:
# These are the exact tags we will use for matching
# Based on what actually exists in the dataset

DIETARY_TAGS = {
    'Vegetarian': 'vegetarian',
    'Vegan': 'vegan',
    'Gluten-Free': 'gluten-free',
    'Dairy-Free': 'dairy-free',
    'Low Sodium': 'low-sodium',
    'Low Carb': 'low-carb',
    'High Protein': 'high-protein',
    'Diabetic Friendly': 'diabetic',
}

CUISINE_TAGS = {
    'Italian': 'italian', 'French': 'french', 'Greek': 'greek',
    'Mexican': 'mexican', 'Asian': 'asian', 'Chinese': 'chinese',
    'Japanese': 'japanese', 'Thai': 'thai', 'Indian': 'indian',
    'Middle Eastern': 'middle-eastern', 'American': 'american',
    'European': 'european', 'African': 'african', 'South American': 'south-american',
}

# Health goals map to the nutritional column(s) most relevant to that goal.
HEALTH_GOALS = {
    'Weight Loss': {
        'column': 'calories [cal]',
        'direction': 'minimize',
        'description': 'lower calorie content',
        # Basis: Weight loss needs an energy deficit
    },
    'Cardiovascular Health': {
        'column': 'sodium [mg]',
        'direction': 'minimize',
        'description': 'lower sodium content',
        # Basis: WHO recommends reducing sodium intake to lower blood pressure
        # and cardiovascular risk (WHO healthy diet guidance).
    },
    'High Protein Diet': {
        'column': 'protein [g]',
        'direction': 'maximize',
        'description': 'higher protein content',
        # Basis: protein adequacy supports muscle maintenance and satiety.
    },
    'Diabetic Management': {
        # Blended: blood sugar response is driven by total carbohydrate load,
        # not sugar alone. Carbohydrate counting is the standard dietary approach
        # for diabetes management, we combine sugars and total carbohydrate.
        'column': 'sugars [g]',
        'secondary_column': 'totalCarbohydrate [g]',
        'direction': 'minimize',
        'description': 'lower sugar and carbohydrate content',
    },
    'Low Fat Diet': {
        'column': 'totalFat [g]',
        'direction': 'minimize',
        'description': 'lower fat content',
        # Basis: reducing total fat reduces energy density.
    },
    'General Healthy Eating': {
        'column': 'nutri_score',
        'direction': 'maximize',
        'description': 'overall nutritional balance',
        # Basis: NutriScore is a validated front-of-pack summary of overall healthiness.
    },
}

# Dietary tags that reinforce each health goal. POSITIVE ranking bonus only
# (never exclusion): a matching tag nudges a recipe up; absence never removes a nutritionally-suitable recipe. Selection relies on actual nutrient values
GOAL_TAGS = {
    'High Protein Diet':      ['high-protein'],
    'Diabetic Management':    ['diabetic', 'low-carb', 'very-low-carbs'],
    'Low Fat Diet':           ['low-fat', 'low-cholesterol'],
    'Weight Loss':            ['low-calorie', 'low-carb', 'low-fat'],
    'Cardiovascular Health':  ['low-sodium', 'low-cholesterol'],
    'General Healthy Eating': ['healthy', 'low-fat'],
}
GOAL_TAG_BONUS = 0.05

print("User profile options defined:")
print(f"  Dietary restrictions: {len(DIETARY_TAGS)}")
print(f"  Cuisine preferences:  {len(CUISINE_TAGS)}")
print(f"  Health goals:         {len(HEALTH_GOALS)}")
print(f"  GOAL TAGS:         {len(GOAL_TAGS)}")

## 4. Test Personas
Six personas, each with a distinct health goal and its own dietary restrictions, allergies, dislikes and cuisine preferences. They are used for the demonstration and as the basis for matching participants in the human evaluation.

In [ ]:
TEST_PERSONAS = {
    'maria': {
        'name': 'Maria',
        'age': 45,
        'health_goal': 'Cardiovascular Health',
        'dietary_restrictions': ['low-sodium'],
        'cuisine_preferences': ['american', 'european'],
        'disliked_ingredients': ['anchovies', 'sardines'],
        'allergies': []
    },
    'james': {
        'name': 'James',
        'age': 32,
        'health_goal': 'High Protein Diet',
        'dietary_restrictions': ['dairy-free'],
        'cuisine_preferences': ['european','american'],
        'disliked_ingredients': [],
        'allergies': ['peanuts']
    },
    'leila': {
        'name': 'Leila',
        'age': 28,
        'health_goal': 'Weight Loss',
        'dietary_restrictions': ['vegan'],
        'cuisine_preferences': ['european', 'american'],
        'disliked_ingredients': [],
        'allergies': []
    },
    'david': {
        'name': 'David',
        'age': 55,
        'health_goal': 'Diabetic Management',
        'dietary_restrictions': ['diabetic', 'low-carb'],
        'cuisine_preferences': ['asian', 'european'],
        'disliked_ingredients': ['blue cheese'],
        'allergies': []
    },
    'sophie': {
        'name': 'Sophie',
        'age': 38,
        'health_goal': 'General Healthy Eating',
        'dietary_restrictions': [],
        'cuisine_preferences': ['european', 'american'],
        'disliked_ingredients': [],
        'allergies': ['tree nuts']
    },
    'aisha': {
        'name': 'Aisha',
        'age': 34,
        'health_goal': 'Low Fat Diet',
        'dietary_restrictions': ['dairy-free'],
        'cuisine_preferences': ['american', 'asian'],
        'disliked_ingredients': ['mayonnaise'],
        'allergies': [] 
    }
}

for key, persona in TEST_PERSONAS.items():
    print(f"\n{persona['name']}, {persona['age']}")
    print(f"  Health goal:    {persona['health_goal']}")
    print(f"  Restrictions:   {persona['dietary_restrictions']}")
    print(f"  Cuisines:       {persona['cuisine_preferences']}")
    print(f"  Allergies:      {persona['allergies']}")
    print(f"  Dislikes:       {persona['disliked_ingredients']}")

## 5. Recommendation Engine
Filter → score → health-quality filter → goal floor → diversity. The design follows the filter-then-rank architecture of Yang et al. (2017) and the weighted multi-objective scoring of De Croon et al. (2025). The specific weights and thresholds are documented design decisions (thesis Appendix 1).

In [ ]:
import ast, re
import difflib

ALLERGEN_EXPANSIONS = {
    'tree nuts': ['almond', 'almonds', 'walnut', 'walnuts', 'cashew', 'cashews',
                  'pecan', 'pecans', 'pistachio', 'pistachios', 'hazelnut',
                  'hazelnuts', 'macadamia', 'brazil nut', 'brazil nuts',
                  'pine nut', 'pine nuts'],
    'peanuts':   ['peanut', 'peanuts', 'groundnut', 'groundnuts'],
    'shellfish': ['shrimp', 'crab', 'lobster', 'crayfish', 'prawn', 'scallop',
                  'clam', 'oyster', 'mussel'],
    'dairy':     ['milk', 'cheese', 'butter', 'cream', 'yogurt', 'yoghurt',
                  'whey', 'lactose', 'ghee'],
    'eggs':      ['egg', 'eggs', 'yolk', 'yolks', 'albumin'],
    'gluten':    ['wheat', 'barley', 'rye', 'flour', 'bread crumb', 'breadcrumb',
                  'semolina', 'spelt'],
    'soy':       ['soy', 'soya', 'tofu', 'tempeh', 'edamame', 'miso', 'tamari']
}

# Terms that break a dietary restriction. Besides plain ingredients, the lists
# include dish words that imply one ('alfredo' implies cream, 'steak' is meat).
DAIRY_TERMS = ['milk', 'cheese', 'butter', 'cream', 'yogurt', 'yoghurt', 'whey',
               'lactose', 'ghee', 'alfredo', 'parmesan', 'mozzarella', 'cheddar',
               'ricotta', 'feta', 'paneer', 'buttermilk', 'custard', 'ice cream']
MEAT_FISH_TERMS = ['clam', 'mussel', 'oyster', 'scallop', 'shellfish', 'seafood',
                   'lobster', 'prawn', 'shrimp', 'crab', 'squid', 'octopus', 'fish',
                   'salmon', 'tuna', 'cod', 'anchovy', 'sardine',
                   'chicken', 'beef', 'steak', 'pork', 'lamb', 'turkey', 'bacon',
                   'ham', 'sausage', 'meat', 'veal', 'duck', 'prosciutto',
                   'pancetta', 'chorizo', 'salami', 'pepperoni', 'lard', 'suet',
                   'gelatin', 'gelatine', 'mince', 'venison', 'bison', 'elk', 'rabbit',
                   'goat', 'game', 'jerky', 'fish sauce',   # 'broth'/'stock' deliberately absent: 'vegetable broth' is vegan; 'chicken broth' is caught by 'chicken'
                   'oyster sauce', 'worcestershire', 'drippings']
DIETARY_VIOLATIONS = {
    'dairy-free':  DAIRY_TERMS,
    'vegetarian':  MEAT_FISH_TERMS,
    'vegan':       MEAT_FISH_TERMS + DAIRY_TERMS + ['egg', 'honey', 'mayonnaise'],
    'gluten-free': ['wheat', 'barley', 'rye', 'flour', 'breadcrumb', 'bread crumb',
                    'semolina', 'spelt', 'pasta'],
}


# Restrictions checked against the ingredient list and title
PREFERENCE_RESTRICTIONS = set(DIETARY_VIOLATIONS.keys())

LEVEL_RESTRICTION_THRESHOLDS = {
    # FSA "low" sodium is < 140 mg per serving
    'low-sodium': [('sodium [mg]', '<', 140)],

    # Documented design threshold
    'low-carb': [('totalCarbohydrate [g]', '<', 30)],

    # Sugars + carbohydrate limits
    'diabetic': [
        ('sugars [g]', '<', 10),
        ('totalCarbohydrate [g]', '<', 45)
    ],
}

# Drinks, condiments, desserts, snacks, side dishes and similar are not main meals
EXCLUDED_TAGS = {
    'beverages', 'cocktails', 'smoothies', 'punch', 'shakes', 'non-alcoholic',
    'brewing', 'herb-and-spice-mixes', 'marinades-and-rubs', 'condiments-etc',
    'sauces', 'jams-and-preserves', 'stocks', 'salad-dressings', 'dips',
    'spreads', 'cake-fillings-and-frostings',
    'desserts', 'snacks', 'candy', 'cookies-and-brownies', 'bar-cookies',
    'frozen-desserts', 'pies-and-tarts', 'infant-baby-friendly', 'toddler-friendly', 'for-large-groups',
    'for-large-groups-holiday-event', 'side-dishes', 'curries'
}

# A recipe needs at least one of these course tags
REQUIRED_COURSE_TAGS = {
    'main-dish', 'breakfast', 'lunch', 'dinner-party', 'salads',
    'soups-stews', 'one-dish-meal', 'breads', 'appetizers', 'main-ingredient',
    'pasta', 'rice', 'casseroles', 'stews', 'stir-fry'
}

# Dish categories for the diversity cap; the first match wins
CATEGORY_TAGS = [
    'pasta', 'salads', 'soups-stews', 'curries', 'stir-fry', 'casseroles',
    'stews', 'rice', 'breads', 'desserts', 'appetizers', 'snacks', 'breakfast',
    'side-dishes', 'main-dish'
]

MIN_CALORIES = 100   # kcal per serving; below this it is not a meal
MAX_PER_CATEGORY = 2
SKIP_RECIPES = {'Tempura', 'Steak and Potato Stir-fry'} # data quality issue

#  recipes that are instructions or crafts, not meals slipped through
# the tag filter ('How to Sprout Lentils', 'German Easter Eggs (Ostereier)').
NON_MEAL_TITLE_RE = re.compile(r'\bhow to\b|\bdye\b|easter egg|\bsprout', re.I)

# the ingredient parser sometimes loses the main ingredient entirely
# ('Venison BBQ' -> ['on your family size)', 'water', ...]; 'Crock Pot Greens and
# Chicken' -> ['$template2$', ...]). 
# Two guards:
#   (1) restriction/allergen/dislike terms are checked against the TITLE as well;
#   (2) recipes whose parsed ingredient list is mostly junk are dropped from the
#       pool - we cannot verify compliance for them, and we cannot explain them.
JUNK_INGREDIENT_RE = re.compile(r'\$template|per person|^\d|^(or|and|a|an|on|it|in)\b|\)$|^\(|packet|quart|liter|'
                                r'\bcup\b|tbsp|tsp|ounce|family size|\bplate\b', re.I)

def ingredient_list_is_reliable(names, min_clean=3, max_junk_share=0.34):
    if not isinstance(names, list) or len(names) == 0:
        return False
    junk = sum(1 for n in names if JUNK_INGREDIENT_RE.search(str(n)))
    return (len(names) - junk) >= min_clean and junk / len(names) <= max_junk_share

def violates(terms, row):
    """Term found in the ingredient list OR the title ."""
    return term_in_ingredients(terms, row['ingredient_names']) or \
           term_in_ingredients(terms, [str(row['title'])])


def term_in_ingredients(terms, names):
    """True if any term occurs as a whole word (singular OR plural) in the
    ingredient list. Replaces raw substring tests, which both over-matched
    ('ham' in 'chamomile') and under-matched ('anchovies' vs 'anchovy fillet')."""
    if not isinstance(names, list) or not names:
        return False
    text = ' ' + ' '.join(str(n).lower() for n in names) + ' '
    for t in terms:
        stem = re.sub(r'(ies|es|s)$', '', t.lower().strip())
        if len(stem) < 3:
            stem = t.lower().strip()
        pat = r'\b' + re.escape(stem) + r'(?:y|ies|es|s)?\b'
        if re.search(pat, text):
            return True
    return False


# ── Helpers ───────────────────────────────────────────────────────────────────

def parse_tags(tags_raw):
    """Convert raw tag data to a clean list of lowercase strings."""
    try:
        if isinstance(tags_raw, list):
            return [t.lower().strip() for t in tags_raw]
        if isinstance(tags_raw, str) and tags_raw.startswith('['):
            return [t.lower().strip() for t in ast.literal_eval(tags_raw)]
        return []
    except:
        return []


def expand_allergens(allergies):
    """Expand generic allergen names to specific ingredient terms."""
    expanded = []
    for a in allergies:
        al = a.lower()
        expanded.extend(ALLERGEN_EXPANSIONS.get(al, [al]))
    return expanded


def get_recipe_category(tags_raw):
    """Identify the primary dish category. Returns 'other' if none match."""
    tags = parse_tags(tags_raw)
    for category in CATEGORY_TAGS:
        if category in tags:
            return category
    return 'other'


# ── Filter (hybrid: strict allergies, relaxed preferences) ────────────────────

def filter_recipes(df, persona, verbose=False):
    """Apply hard filters. Allergies strict, dietary preferences relaxed."""
    filtered = df.copy()

    # Rule 0: real food only
    def is_real_food(t):
        tags = parse_tags(t)
        if not tags or any(x in EXCLUDED_TAGS for x in tags):
            return False
        return any(x in REQUIRED_COURSE_TAGS for x in tags)
    filtered = filtered[filtered['tags'].apply(
        lambda t: is_real_food(t) if pd.notna(t) else False)]


    # Rule 1: minimum calories
    filtered = filtered[filtered['calories [cal]'] >= MIN_CALORIES]

    # Rule 1a : drop instruction/craft 'recipes'
    filtered = filtered[~filtered['title'].astype(str).str.contains(NON_MEAL_TITLE_RE)]

    # Rule 1c : drop recipes whose ingredient list is too broken to verify
    filtered = filtered[filtered['ingredient_names'].apply(ingredient_list_is_reliable)]

    
    # Rule 1b: substantial meal floor - a real main dish has some protein.
    # Prevents minimize-nutrient goals (e.g. low-fat) from surfacing plain
    # starches (rice, potato) instead of balanced low-fat meals.
    # Documented design threshold.
    MIN_PROTEIN = 5  # g per serving - below this is a side/plain starch, not a meal
    filtered = filtered[filtered['protein [g]'] >= MIN_PROTEIN]

        # Rule 2: dietary restrictions
    for r in persona['dietary_restrictions']:
        if r in PREFERENCE_RESTRICTIONS:
            # Ingredient-based restriction (vegan, vegetarian, dairy-free, gluten-free)
            violations = DIETARY_VIOLATIONS[r]
            
            def passes(row):
                # the 'tagged vegan -> pass' shortcut is gone. Tags are a
                # folksonomy and were letting meat dishes through; the ingredient
                # list (and title) is the ground truth we can actually verify.
                return not violates(violations, row)
            filtered = filtered[filtered.apply(passes, axis=1)]

        elif r in LEVEL_RESTRICTION_THRESHOLDS:
            # Value-based restriction (low-sodium, low-carb, diabetic):
            # hard-filter on the actual nutrient values using cited thresholds.
            for col, op, val in LEVEL_RESTRICTION_THRESHOLDS[r]:
                if op == '<':
                    filtered = filtered[filtered[col] < val]
                elif op == '<=':
                    filtered = filtered[filtered[col] <= val]

        else:
            # Fallback: require the restriction as a tag
            filtered = filtered[filtered['tags'].apply(
                lambda t: r in parse_tags(t) if pd.notna(t) else False)]


    # Rule 3: allergens (strict)
    if persona['allergies']:
        expanded = expand_allergens(persona['allergies'])
        filtered = filtered[~filtered.apply(lambda row: violates(expanded, row), axis=1)]

    # Rule 4: dislikes
    if persona['disliked_ingredients']:
        filtered = filtered[~filtered.apply(lambda row: violates(persona['disliked_ingredients'], row), axis=1)]

    if verbose:
        print(f"  Filtered pool: {len(filtered):,} recipes")
    return filtered


# ── Score (health 60% goal / 40% WHO, then cuisine and rating) ────────────────

def score_recipes(df, persona):
    """Score recipes. health = 70% specific goal + 30% overall WHO.
    For goals with a secondary column (e.g. diabetic: sugars + carbs),
    the goal sub-score averages both normalised columns."""
    scored = df.copy()

    cfg = HEALTH_GOALS[persona['health_goal']]
    col, direction = cfg['column'], cfg['direction']

    def normalise_col(column, direction):
        if column not in scored.columns:
            return 0.5
        lo, hi = scored[column].min(), scored[column].max()
        rng = hi - lo
        if rng <= 0:
            return 0.5
        if direction == 'minimize':
            return 1 - ((scored[column] - lo) / rng)
        return (scored[column] - lo) / rng

    goal = normalise_col(col, direction)

    # If there is a secondary column (diabetic: sugars + carbohydrate), blend it
    if 'secondary_column' in cfg:
        goal_secondary = normalise_col(cfg['secondary_column'], direction)
        goal = (goal + goal_secondary) / 2

    wlo, whi = scored['who_score'].min(), scored['who_score'].max()
    wrng = whi - wlo
    who_norm = (scored['who_score'] - wlo) / wrng if wrng > 0 else 0.5

    scored['health_score'] = (goal * 0.7) + (who_norm * 0.3)

    def cuisine(t):
        tags = parse_tags(t)
        return 1.0 if tags and any(c in tags for c in persona['cuisine_preferences']) else 0.0
    scored['cuisine_score'] = scored['tags'].apply(cuisine)

    scored['rating_score'] = scored['average_rating'] / 5.0

    def goal_tag_match(t):
        tags = parse_tags(t)
        goal_tags = GOAL_TAGS.get(persona['health_goal'], [])
        return GOAL_TAG_BONUS if tags and any(gt in tags for gt in goal_tags) else 0.0
    scored['goal_tag_bonus'] = scored['tags'].apply(goal_tag_match)   

    # ── Universal nutritional safety floor ───────────────────────────────
    # Some nutrients pose health risks above certain levels for ALL users,
    # regardless of their stated goal. Sodium is the key example - WHO
    # guidance recommends everyone limit sodium intake. Recipes exceeding
    # a high per-serving sodium threshold are down-weighted so they cannot
    # dominate the top recommendations for any persona.
    SODIUM_CEILING = 500  # mg per serving - above this is high for one meal

    scored['safety_multiplier'] = scored['sodium [mg]'].apply(
        lambda s: 0.7 if (pd.notna(s) and s > SODIUM_CEILING) else 1.0
    )


    scored['final_score'] = (
        (scored['health_score'] * 0.65) +
        (scored['cuisine_score'] * 0.15) +
        (scored['rating_score'] * 0.20) +
        scored['goal_tag_bonus']
    ) * scored['safety_multiplier']
    
    return scored.sort_values('final_score', ascending=False)


# ── Diversity (max 2 per category) ────────────────────────────────────────────

def _norm_title(t):
    t = re.sub(r'\(.*?\)', '', str(t).lower())        # drop parentheticals
    return re.sub(r'[^a-z ]', '', t).strip()

def apply_diversity(scored_df, n=5, max_per_category=MAX_PER_CATEGORY, title_sim=0.75):
    """Build a diverse top-n list: cap per category AND skip near-duplicate titles ."""
    selected, counts, titles = [], {}, []
    for _, row in scored_df.iterrows():
        cat = get_recipe_category(row['tags'])
        nt = _norm_title(row['title'])
        if any(difflib.SequenceMatcher(None, nt, t).ratio() >= title_sim for t in titles):
            continue
        if counts.get(cat, 0) < max_per_category:
            selected.append(row); titles.append(nt)
            counts[cat] = counts.get(cat, 0) + 1
        if len(selected) == n:
            break
    return pd.DataFrame(selected)


# ── Full pipeline ─────────────────────────────────────────────────────────────

def apply_nutriscore_filter(scored_df, n=5):
    """
    Prefer NutriScore A and B; include C only if fewer than n A/B recipes exist.
    Never include D or E.

    Grounded in the HUMMUS authors' health-aware post-processing, which filters
    recommendations by a NutriScore threshold (filtering_util.filter_recipes_by_threshold),
    and in the study's health-focused framing: every recommendation should be at
    least reasonably healthy. A-B is the primary cutoff; C is a documented fallback
    used only when A-B cannot fill the recommendation slate.
    """
    ab = scored_df[scored_df['nutri_grade'].isin(['A', 'B'])]
    if len(ab) >= n:
        return ab
    # Fallback: not enough A/B, so allow C to fill the remaining slots
    c = scored_df[scored_df['nutri_grade'] == 'C']
    return pd.concat([ab, c])

def apply_goal_floor(scored_df, persona, percentile=0.85, min_keep=15):
    """
    Restrict to recipes that genuinely match the goal nutrient (a high percentile
    floor, because the goal-nutrient distribution is skewed), then let the existing
    composite score rank the survivors.

    Rationale (documented design decision): the floor guarantees goal-fit so the
    explanation ('selected to support your <goal>') is honest, while ranking the
    survivors by the composite score - which includes overall nutritional balance -
    prevents degenerate single-nutrient results (e.g. fat-free confectionery for a
    low-fat goal). Goal-fit and balance are thus both enforced.
    """
    cfg = HEALTH_GOALS[persona['health_goal']]
    col, direction = cfg['column'], cfg['direction']
    if col not in scored_df.columns:
        return scored_df

    if direction == 'minimize':
        cutoff = scored_df[col].quantile(1 - percentile)
        kept = scored_df[scored_df[col] <= cutoff]
    else:
        cutoff = scored_df[col].quantile(percentile)
        kept = scored_df[scored_df[col] >= cutoff]

    if len(kept) < min_keep:
        if direction == 'minimize':
            kept = scored_df.nsmallest(min_keep, col)
        else:
            kept = scored_df.nlargest(min_keep, col)

    # Keep the existing composite-score order (do NOT re-sort by goal nutrient) -
    # this preserves overall balance and prevents degenerate single-nutrient picks.
    return kept.sort_values('final_score', ascending=False)


def get_recommendations(persona, n=5, diverse=True, df=None):
    """
    Complete pipeline: filter -> score -> NutriScore health filter -> diversify.
    Returns top-n recommended recipes for a persona.
    Pass df to override the default clean_recipes dataset.
    """
    if df is None:
        df = clean_recipes
    filtered = filter_recipes(df, persona, verbose=False)
    scored   = score_recipes(filtered, persona)
    scored   = apply_nutriscore_filter(scored, n=n) 
    scored   = apply_goal_floor(scored, persona)     
    if diverse:
        return apply_diversity(scored, n=n)
    return scored.head(n)


def get_all_recommendations(personas, n=5, gamma=0.15):
    """
    Generate recommendations for all personas with an inter-list diversity penalty.

    A recipe already recommended to another persona has its score penalised by
    gamma per prior appearance, so each persona receives more distinctive
    recommendations and the same broadly-healthy recipes do not dominate every list.

    Rationale (documented design decision): reduces cross-list redundancy, a form
    of aggregate/inter-list diversity in recommender systems (Starke et al. 2021,
    multi-list food recommendation). gamma is a documented design parameter.
    """
    appearance_count = {}   # recipe_title -> how many personas already got it
    all_recs = {}

    for pk, persona in personas.items():
        filtered = filter_recipes(clean_recipes, persona, verbose=False)
        scored   = score_recipes(filtered, persona)
        scored   = apply_nutriscore_filter(scored, n=n)
        scored   = apply_goal_floor(scored, persona)

        def _keep(row):
            if row['title'] in SKIP_RECIPES:
                return False
            ings_raw = row.get('ingredients_display')
            ings = ast.literal_eval(ings_raw) if isinstance(ings_raw, str) else ings_raw
            return len(select_key_ingredients(ings, row['title'])) > 0
        scored = scored[scored.apply(_keep, axis=1)]

        # Apply inter-list diversity penalty
        scored = scored.copy()
        scored['penalized_score'] = scored.apply(
            lambda r: r['final_score'] - gamma * appearance_count.get(r['title'], 0),
            axis=1
        )
        scored = scored.sort_values('penalized_score', ascending=False)

        recs = apply_diversity(scored, n=n)
        all_recs[pk] = recs

        # Record appearances for subsequent personas
        for title in recs['title']:
            appearance_count[title] = appearance_count.get(title, 0) + 1

    return all_recs


# ── Quick test ────────────────────────────────────────────────────────────────

print("Pipeline ready. Testing all personas:\n")
for key, persona in TEST_PERSONAS.items():
    recs = get_recommendations(persona)
    titles = [row['title'] for _, row in recs.iterrows()]
    print(f"{persona['name']:8} ({persona['health_goal']}):")
    for t in titles:
        print(f"    - {t}")
    print()

In [ ]:
# The nutrient values behind each list, to check that the goal is followed
for pk, persona in TEST_PERSONAS.items():
    recs = get_recommendations(persona, n=5)
    cfg = HEALTH_GOALS[persona['health_goal']]
    print(f"\n{persona['name']} ({persona['health_goal']} → {cfg['direction']} {cfg['column']}):")
    for _, r in recs.iterrows():
        print(f"    {r['title'][:34]:34} [{r['nutri_grade']}] "
              f"prot={r['protein [g]']:.0f} cal={r['calories [cal]']:.0f} "
              f"fat={r['totalFat [g]']:.0f} carb={r['totalCarbohydrate [g]']:.0f} "
              f"sug={r['sugars [g]']:.0f} sod={r['sodium [mg]']:.0f}")

## 6. Inter-List Diversity Check
Measure recipe overlap across the personas' lists.

In [ ]:
from collections import Counter

def measure_persona_overlap(personas, n=5):
    """Measure how many recipes are shared across multiple personas' top-n.
    Pure engine logic - no API calls."""
    # Collect each persona's recipe titles
    persona_recipes = {}
    for key, persona in personas.items():
        recs = get_recommendations(persona, n=n)
        persona_recipes[persona['name']] = set(recs['title'].tolist())

    # Count how many personas each recipe appears in
    all_titles = []
    for titles in persona_recipes.values():
        all_titles.extend(titles)
    title_counts = Counter(all_titles)

    # Recipes appearing in more than one persona
    shared = {t: c for t, c in title_counts.items() if c > 1}

    total_unique = len(title_counts)
    total_slots = sum(len(t) for t in persona_recipes.values())

    print(f"Total recommendation slots: {total_slots} ({len(personas)} personas x {n})")
    print(f"Unique recipes across all personas: {total_unique}")
    print(f"Recipes shared by 2+ personas: {len(shared)}")
    print(f"Overlap rate: {(total_slots - total_unique) / total_slots * 100:.1f}%")

    if shared:
        print(f"\nShared recipes (appear in multiple personas):")
        for title, count in sorted(shared.items(), key=lambda x: -x[1]):
            who_has = [name for name, titles in persona_recipes.items() if title in titles]
            print(f"  [{count}x] {title[:45]}")
            print(f"        -> {', '.join(who_has)}")
    else:
        print("\nNo overlap - every persona has completely distinct recipes.")

    return persona_recipes

print("="*70)
print("OVERLAP TEST - CURRENT PERSONAS")
print("="*70)
_ = measure_persona_overlap(TEST_PERSONAS)

## 7. Explanation Generation
Generate one number-free explanation per recommendation with a single pinned model (Gemma 4 31B), using only the verified facts for that recipe and persona.

In [ ]:
import time, os
from openai import OpenAI

# ── SINGLE PINNED MODEL ───────────────────────────────────────────────────────
# The entire frozen explanation set come from ONE model, so the only thing
# differing between explanations is the persona, not the generator. If the model
# is capped for the day, generation STOPS and resumes later on the SAME model —
# it never falls back to a different one.
GENERATION_MODEL = 'jwwqblcgkizlhxjjbkcp'  # Gemma4 31B (Lehmus)
EXPLANATIONS_FILE = 'data/explanations.csv'

# Human-readable names for Lehmus model IDs -- used so model_used in the CSV
LEHMUS_MODEL_NAMES = {
    'jwwqblcgkizlhxjjbkcp': 'Gemma4 31B',
    'ctraxdkjuwfrlivzrhgt': 'Qwen3.6-35B-A3B',
    'azsydsttjnlbfjbgqnwd': 'GPT OSS 120B',
}

client = OpenAI(
    api_key=os.getenv('LEHMUS_API_KEY'),
    base_url=os.getenv('LEHMUS_BASE_URL'),
)


def generate_single_model(prompt, model_name=GENERATION_MODEL):
    """
    Generate ONE explanation with ONE pinned model.
    Returns (text, display_name) on success.
    Returns (None, display_name) if capped (429) or persistently overloaded (503)
    so the caller stops and resumes later. Other errors are raised.
    """
    display_name = LEHMUS_MODEL_NAMES.get(model_name, model_name)
    max_retries = 3
    for attempt in range(max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
            )
            return response.choices[0].message.content.strip(), display_name

        except Exception as e:
            status = getattr(e, 'status_code', None)
            error_str = str(e)

            # 503: temporary overload -> back off and retry the SAME model
            if status == 503 or '503' in error_str or 'UNAVAILABLE' in error_str:
                if attempt < max_retries:
                    sleep_time = 2 ** (attempt + 1)
                    print(f"    {display_name} busy (503). Retry in {sleep_time}s "
                          f"({attempt + 1}/{max_retries})...")
                    time.sleep(sleep_time)
                    continue
                print(f"    {display_name} still overloaded after {max_retries} retries. Stopping.")
                return None, display_name

            # 429: daily cap / rate limit -> stop, do NOT fall back to another model
            if status == 429 or '429' in error_str or 'rate limit' in error_str.lower():
                print(f"    {display_name} hit its rate limit. Resume later on the same model.")
                return None, display_name

            # anything else is a genuine failure -> surface it
            raise

    return None, display_name


def build_explanation_dataset(personas, n_per_persona=5, verbose=True):
    """
    Resumable SINGLE-MODEL generation using the diversity-aware recommendations
    (get_all_recommendations), so the frozen set matches exactly what participants
    see. Number-free, goal-qualitative explanations (Design 1).
    Re-reads the CSV each iteration so multi-day resume is safe.
    Saves after EVERY successful explanation. Stops cleanly when capped.
    """
    os.makedirs(os.path.dirname(EXPLANATIONS_FILE), exist_ok=True)

    # Generate ALL personas' recommendations together (applies inter-list diversity)
    all_recs = get_all_recommendations(personas, n=n_per_persona)

    for persona_key, persona in personas.items():
        recs = all_recs[persona_key]
        # Reference pool for the goal-rank fact: the same filtered, scored and
        # Nutri-Score-filtered set that apply_goal_floor ranks, so every
        # recommended recipe is at or above the 85th percentile
        pool = apply_nutriscore_filter(score_recipes(filter_recipes(clean_recipes, persona), persona), n=5)

        for idx, (_, recipe_row) in enumerate(recs.iterrows()):
            title = recipe_row['title']

            if os.path.exists(EXPLANATIONS_FILE):
                done_df = pd.read_csv(EXPLANATIONS_FILE)
                done_keys = set(zip(done_df['persona_key'], done_df['recipe_title']))
            else:
                done_df = pd.DataFrame()
                done_keys = set()

            if (persona_key, title) in done_keys:
                if verbose:
                    print(f"  [skip] {persona['name']}: {title}")
                continue

            # never explain a non-compliant recipe; log it instead
            bad = compliance_violations(recipe_row, persona)
            if bad:
                print(f"  [BLOCKED] {persona['name']}: {title} violates {bad} - fix the engine, not the prompt")
                continue
            facts = extract_grounding_facts(recipe_row, persona, pool_df=pool, opening_style=idx)
            prompt = build_grounded_prompt(facts)
            explanation, model_used = generate_single_model(prompt)

            if explanation is None:
                print(f"\n  {GENERATION_MODEL} unavailable/capped. Progress saved.")
                print(f"  Re-run this cell later to resume on the SAME model.\n")
                return

            new_row = pd.DataFrame([facts_to_row(persona_key, persona, recipe_row, facts, explanation, model_used)])

            combined = pd.concat([done_df, new_row], ignore_index=True)
            combined.to_csv(EXPLANATIONS_FILE, index=False)

            if verbose:
                print(f"  [done] {persona['name']}: {title}  ({model_used})")

            time.sleep(2)

    if os.path.exists(EXPLANATIONS_FILE):
        final = pd.read_csv(EXPLANATIONS_FILE)
        models = final['model_used'].unique()
        print(f"\n  Complete. Total: {len(final)} explanations.")
        print(f"  Models used: {list(models)}")
        if len(models) > 1:
            print("  WARNING: more than one model — investigate before freezing.")


print(f"Ready. Pinned model: {LEHMUS_MODEL_NAMES.get(GENERATION_MODEL, GENERATION_MODEL)}")


### Fact extraction
Build the verified facts for each recipe and persona: five core facts that every explanation draws on, and up to two optional ones.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# GROUNDED FACT EXTRACTION  (Design 1b: recipe-distinctive, still number-free)
# ═════════════════════════════════════════════════════════════════════════════
#
#   fact                 source                                     claim it licenses
#   ─────────────────────────────────────────────────────────────────────────────
#   key_ingredients      recipe ingredient list (corpus)            "made with lentils, garlic and tomato"
#   dish_type            recipe tags (corpus)                       "a hearty soup"
#   goal_rank_tier       percentile of goal nutrient within the     "among the strongest options for lower
#                        persona's filtered candidate pool           sodium content of the recipes that fit
#                        (engine computation - corpus-RELATIVE,       your profile"
#                        not an absolute FSA level)
#   matched_cuisine      intersection of recipe tags & persona      "a European dish"
#   matched_goal_tags    recipe tags the author applied             "tagged low-sodium by its author"
#   highly_rated         average_rating >= 4.5 & >= 10 ratings      "well rated by other cooks"
#   balance_desc         NutriScore grade (unchanged from Design 1)  "a healthy choice (NutriScore B)"

import re

def describe_balance_from_nutri(grade):
    """Plain language healthiness phrase derived from the NutriScore grade.
    NutriScore is fixed and dataset-independent (per-100g, computed by the HUMMUS
    scoring code), unlike the corpus-relative WHO score, so it anchors the
    healthiness descriptor."""
    phrases = {
        'A': "a nutritionally strong choice",
        'B': "a healthy choice",
        'C': "a moderate choice",
        'D': "a lower-scoring choice that is reasonable in moderation",
        'E': "an indulgent choice, best enjoyed occasionally",
    }
    return phrases.get(grade, "a reasonable choice")

def describe_nutri_grade(grade):
    return f"NutriScore {grade}"


# ── Key ingredients ───────────────────────────────────────────────────────────
#  choose by INFORMATIVENESS, not list order. The regeneration named
# 'onion and bell pepper' for a chili whose black beans sat in position 11, and
# 'carrots and celery' for Chicken Lentil Soup. Ingredients are ranked in tiers:
#   1 proteins / legumes / dairy / nuts   3 vegetables / fruit / mushrooms
#   2 grains / pasta / rice / potato      4 aromatics, herbs, spices, sauces
# Pantry items and parsing junk are dropped. Ingredients already in the title
# are demoted (kept only if nothing better exists), not banned.
PANTRY_TERMS = {
    'salt', 'pepper', 'water', 'oil', 'ice', 'sugar', 'flour', 'vanilla',
    'baking powder', 'baking soda', 'cooking spray', 'cream of tartar', 'meal',
}
_PANTRY_RE = re.compile(
    r'^(salt|pepper|water|oil|ice|sugar|flour|vanilla|shortening|margarine|butter)\b|'
    r'\b(salt|pepper|peppercorns|water|oil|sugar|flour|vanilla|baking (powder|soda)|cooking spray)$', re.I)
_TIER = [
    (1, r'bean|lentil|pea\b|peas\b|chickpea|garbanzo|dal\b|tofu|tempeh|egg|cheese|yogurt|yoghurt|'
        r'chicken|turkey|beef|steak|pork|lamb|ham\b|bacon|sausage|fish|salmon|tuna|cod\b|shrimp|prawn|'
        r'crab|venison|duck|nut\b|nuts\b|almond|walnut|cashew|peanut|seed'),
    (2, r'rice|pasta|spaghetti|noodle|barley|bulgur|quinoa|oat|couscous|potato|bread|tortilla|'
        r'pita|semolina|polenta|wheat|dosa|urad'),
    (3, r'tomato|spinach|kale|escarole|cabbage|broccoli|cauliflower|zucchini|squash|carrot|'
        r'mushroom|eggplant|aubergine|corn|artichoke|asparagus|avocado|cucumber|greens|'
        r'apple|orange|lemon\b|lime\b|berry|berries|banana|mango|pineapple|raisin|coconut'),
    (4, r'onion|garlic|celery|ginger|chili|chilli|pepper|herb|basil|parsley|cilantro|thyme|'
        r'rosemary|oregano|cumin|curry|paprika|turmeric|saffron|spice|sauce|vinegar|wine|'
        r'broth|stock|juice|syrup|mustard|paste|seasoning|powder'),
]
_PREFIX_RE = re.compile(
    r'^(chopped|diced|sliced|minced|grated|shredded|crushed|fresh|dried|large|medium|small|'
    r'thinly sliced|finely chopped|boneless|skinless|ground|cooked|frozen|canned|ripe|'
    r'stalks?|bunch(es)? of|bunch|package|head|jar|box|can|slices|some|a few|plate)\s+', re.I)

def _tier(s):
    for t, pat in _TIER:
        if re.search(pat, s):
            return t
    return 5

def select_key_ingredients(ingredients_display, title, k=3):
    """Pick up to k informative ingredient names (verified corpus data)."""
    if not isinstance(ingredients_display, list):
        return []
    t = title.lower()
    cands = []
    for pos, ing in enumerate(ingredients_display):
        s = ing.lower().strip()
        if not s or JUNK_INGREDIENT_RE.search(s) or _PANTRY_RE.search(s) or s in PANTRY_TERMS:
            continue
        for _ in range(2):
            s = _PREFIX_RE.sub('', s).strip()
        if len(s) < 3 or s in PANTRY_TERMS:
            continue
        in_title = any(w in t for w in s.split() if len(w) > 3)
        cands.append((_tier(s), 1 if in_title else 0, pos, s))
    cands.sort()
    out, heads = [], set()
    for _, _, _, s in cands:
        head = re.sub(r's$', '', s.split()[-1])       # one per head noun: 'orange'/'oranges', 'lemon'/'lemon juice'
        if s not in out and head not in heads:
            out.append(s); heads.add(head)
        if len(out) == k:
            break
    return out


# ── Compliance guard  ─────────────────────────────────────────────────────
def compliance_violations(recipe_row, persona):
    """Independent re-check before we spend an API call: never explain a recipe
    that breaks the persona's hard constraints (the prompt asserts 'the recipe
    complies'). Returns a list of (constraint, terms) hits; empty = OK."""
    hits = []
    for r in persona['dietary_restrictions']:
        if r in DIETARY_VIOLATIONS and violates(DIETARY_VIOLATIONS[r], recipe_row):
            hits.append((r, 'dietary'))
    if persona['allergies'] and violates(expand_allergens(persona['allergies']), recipe_row):
        hits.append(('allergy', 'allergen'))
    if persona['disliked_ingredients'] and violates(persona['disliked_ingredients'], recipe_row):
        hits.append(('dislike', 'dislike'))
    return hits


# ── Dish type from tags ───────────────────────────────────────────────────────
DISH_TYPE_LABELS = {
    'pasta': 'pasta dish', 'salads': 'salad', 'soups-stews': 'soup or stew',
    'stews': 'stew', 'curries': 'curry', 'stir-fry': 'stir-fry',
    'casseroles': 'casserole', 'rice': 'rice dish', 'breads': 'bread',
    'breakfast': 'breakfast dish', 'appetizers': 'appetizer',
    'main-dish': 'main dish', 'side-dishes': 'side dish',
}
DISH_TYPE_PRIORITY = ['soups-stews', 'stews', 'salads', 'curries', 'stir-fry', 'casseroles',
                      'pasta', 'rice', 'breads', 'breakfast', 'appetizers', 'main-dish', 'side-dishes']
def describe_dish_type(tags_raw):
    # own priority order (soup before pasta) - get_recipe_category's order is
    # tuned for the diversity cap, not for naming the dish.
    tags = parse_tags(tags_raw)
    cat = next((c for c in DISH_TYPE_PRIORITY if c in tags), 'other')
    return DISH_TYPE_LABELS.get(cat, 'dish')


# ── Goal-rank tier (corpus-relative, computed by the engine) ─────────────────
#  the LLM dropped the direction word in 4/30 ('one of the top choices for
# sodium content' reads as HIGHEST sodium). The rank phrase now carries the
# direction inside a superlative that cannot be shortened without changing meaning.
GOAL_RANK_PHRASE = {
    'Cardiovascular Health':  'lowest in sodium',
    'Weight Loss':            'lowest in calories',
    'Low Fat Diet':           'lowest in fat',
    'Diabetic Management':    'lowest in sugar and carbohydrate',
    'High Protein Diet':      'highest in protein',
    'General Healthy Eating': 'best in overall nutritional balance',
}

# Optional extra facts, number-free and computed against the same pool:
#   secondary_strength  another nutrient on which the recipe beats at least 85% of the pool
#   caveat              a goal-relevant nutrient on which it beats fewer than 35% of the
#                       pool, so the explanation can mention a trade-off honestly
#   quick_to_make       duration of QUICK_MINUTES or less
INCLUDE_CAVEAT = True
QUICK_MINUTES  = 30
NUTRIENT_DIRS = {   # column -> (direction, phrase)
    'sodium [mg]':          ('minimize', 'lowest in sodium'),
    'totalFat [g]':         ('minimize', 'lowest in fat'),
    'saturatedFat [g]':     ('minimize', 'lowest in saturated fat'),
    'sugars [g]':           ('minimize', 'lowest in sugar'),
    'calories [cal]':       ('minimize', 'lowest in calories'),
    'protein [g]':          ('maximize', 'highest in protein'),
    'dietaryFiber [g]':     ('maximize', 'highest in fibre'),
}
CAVEAT_PHRASE = {   # how to word a below-median position, number-free and neutral
    'sodium [mg]':      'it is not among the lower-sodium options',
    'totalFat [g]':     'it is not among the lower-fat options',
    'saturatedFat [g]': 'it is not among the lower-saturated-fat options',
    'sugars [g]':       'it is not among the lower-sugar options',
    'calories [cal]':   'it is not among the lower-calorie options',
    'protein [g]':      'it is not among the higher-protein options',
    'dietaryFiber [g]': 'it is not among the higher-fibre options',
}

CAVEAT_RELEVANT = {
    'Cardiovascular Health':  ['saturatedFat [g]', 'totalFat [g]', 'sugars [g]'],
    'Weight Loss':            ['totalFat [g]', 'sugars [g]', 'saturatedFat [g]'],
    'Low Fat Diet':           ['saturatedFat [g]', 'sodium [mg]', 'sugars [g]'],
    'Diabetic Management':    ['saturatedFat [g]', 'sodium [mg]', 'totalFat [g]'],
    'High Protein Diet':      ['saturatedFat [g]', 'sodium [mg]', 'sugars [g]'],
    'General Healthy Eating': ['sodium [mg]', 'saturatedFat [g]', 'sugars [g]'],
}
CAVEAT_THRESHOLD = 0.35

def _pct(col, direction, recipe_row, pool_df):
    if col not in pool_df.columns or pd.isna(recipe_row.get(col)):
        return None
    vals = pool_df[col].dropna(); x = float(recipe_row[col])
    return float((vals > x).mean()) if direction == 'minimize' else float((vals < x).mean())

def secondary_and_caveat(recipe_row, pool_df, persona, primary_col):
    """The strongest secondary nutrient and the weakest relevant one, as phrases."""
    if pool_df is None:
        return None, None
    best, worst = (None, -1), (None, 2)
    relevant = CAVEAT_RELEVANT.get(persona['health_goal'], [])
    for col, (direction, phrase) in NUTRIENT_DIRS.items():
        if col == primary_col:
            continue
        p = _pct(col, direction, recipe_row, pool_df)
        if p is None:
            continue
        if p >= 0.85 and p > best[1]:
            best = (phrase, p)
        if col in relevant and p < CAVEAT_THRESHOLD and p < worst[1]:
            worst = (CAVEAT_PHRASE[col], p)
    return best[0], (worst[0] if INCLUDE_CAVEAT else None)

def is_quick(recipe_row):
    try:
        return float(recipe_row['duration']) <= QUICK_MINUTES
    except (KeyError, TypeError, ValueError):
        return False


def goal_percentile(recipe_row, pool_df, persona):
    """Fraction of the persona's FILTERED candidate pool that this recipe beats
    on the goal nutrient (1.0 = best in pool). Corpus-relative by design."""
    cfg = HEALTH_GOALS[persona['health_goal']]
    col, direction = cfg['column'], cfg['direction']
    if col not in pool_df.columns or pool_df[col].notna().sum() == 0:
        return None
    vals = pool_df[col].dropna()
    x = float(recipe_row[col])
    if direction == 'minimize':
        return round(float((vals > x).mean()), 3)
    return round(float((vals < x).mean()), 3)

def describe_goal_rank(pct):
    """Number-free tier. Every recommended recipe is >= 0.85 by construction
    (apply_goal_floor); the tiers just distinguish 'very top' from 'top'."""
    if pct is None:            return None
    if pct >= 0.95:            return "among the very best options"
    if pct >= 0.85:            return "among the strongest options"
    if pct >= 0.60:            return "a better than average option"
    return None   # below 0.60: make no rank claim at all


# ── Cuisine / goal-tag / rating facts ─────────────────────────────────────────
CUISINE_LABELS = {v: k for k, v in CUISINE_TAGS.items()}   # 'european' -> 'European'

def matched_cuisines(tags_raw, persona):
    tags = set(parse_tags(tags_raw))
    return [CUISINE_LABELS.get(cz, cz.title()) for cz in persona['cuisine_preferences'] if cz in tags]

def matched_goal_tags(tags_raw, persona):
    tags = set(parse_tags(tags_raw))
    return [gt for gt in GOAL_TAGS.get(persona['health_goal'], []) if gt in tags]

def is_highly_rated(recipe_row, min_rating=4.5, min_n=10):
    try:
        return float(recipe_row['average_rating']) >= min_rating and \
               float(recipe_row['number_of_ratings']) >= min_n
    except (KeyError, TypeError, ValueError):
        return False


# ── Fact extraction ───────────────────────────────────────────────────────────
def extract_grounding_facts(recipe_row, persona, pool_df=None, opening_style=0):
    """
    Extract verified facts for the explanation, the judge and the display panel.
    pool_df: the persona's filtered candidate pool (output of filter_recipes),
             needed for the corpus-relative goal-rank fact. If omitted, no rank
             claim is made.
    Raw numeric values are kept for the judge and the display panel only - they
    are never placed in the explanation text.
    """
    goal_config = HEALTH_GOALS[persona['health_goal']]
    nutri = recipe_row.get('nutri_grade', 'N/A')
    if goal_config['column'] == 'nutri_score':
        pct = None
        goal_rank_desc = {'A': 'among the very best options', 'B': 'among the strongest options'}.get(nutri)
        goal_rank_phrase = 'highest-graded for overall nutritional balance'
    else:
        pct = goal_percentile(recipe_row, pool_df, persona) if pool_df is not None else None
        goal_rank_desc = describe_goal_rank(pct)
        goal_rank_phrase = GOAL_RANK_PHRASE.get(persona['health_goal'], goal_config['description'])

    facts = {
        'title': recipe_row['title'],
        'health_goal': persona['health_goal'],
        'nutrient_focus': goal_config['description'],

        # Raw values - judge and display panel only
        'calories':        round(float(recipe_row['calories [cal]']), 1),
        'protein_g':       round(float(recipe_row['protein [g]']), 1),
        'sodium_mg':       round(float(recipe_row['sodium [mg]']), 1),
        'total_fat_g':     round(float(recipe_row['totalFat [g]']), 1),
        'saturated_fat_g': round(float(recipe_row['saturatedFat [g]']), 1),
        'sugars_g':        round(float(recipe_row['sugars [g]']), 1),
        'fiber_g':         round(float(recipe_row['dietaryFiber [g]']), 1),
        'who_score':       round(float(recipe_row['who_score']), 3),
        'nutri_grade':     nutri,

        # Healthiness signal used IN the explanation (NutriScore-based only)
        'balance_desc': describe_balance_from_nutri(nutri),
        'nutri_gloss':  describe_nutri_grade(nutri),

        # Recipe-distinctive facts used IN the explanation (all verified)
        'key_ingredients':   select_key_ingredients(recipe_row.get('ingredients_display', []), recipe_row['title']),
        'dish_type':         describe_dish_type(recipe_row['tags']),
        'goal_percentile':   pct,
        'goal_rank_desc':    goal_rank_desc,
        'goal_rank_phrase':  goal_rank_phrase,
        'secondary_strength': (secondary_and_caveat(recipe_row, pool_df, persona, goal_config['column'])[0] if pool_df is not None else None),
        'caveat':             (secondary_and_caveat(recipe_row, pool_df, persona, goal_config['column'])[1] if pool_df is not None else None),
        'quick_to_make':      is_quick(recipe_row),
        'matched_cuisine':   matched_cuisines(recipe_row['tags'], persona),
        'matched_goal_tags': matched_goal_tags(recipe_row['tags'], persona),
        'highly_rated':      is_highly_rated(recipe_row),
        # rotate the requested opening so 10/30 explanations don't start 'You might enjoy'
        'opening_style':     opening_style % 3,

        # User context
        'dietary_restrictions': list(persona['dietary_restrictions']),
        'cuisine_preferences':  list(persona['cuisine_preferences']),
        'cuisine_matched':      bool(matched_cuisines(recipe_row['tags'], persona)),
    }
    return facts


def facts_to_row(persona_key, persona, recipe_row, facts, explanation, model_used):
    """One CSV row. The exact facts shown to the LLM are persisted so the judge
    later verifies against the SAME ground truth the generator received."""
    return {
        'persona_key': persona_key,
        'persona_name': persona['name'],
        'health_goal': persona['health_goal'],
        'recipe_title': recipe_row['title'],
        'nutri_grade': facts['nutri_grade'],
        'who_score': facts['who_score'],
        'sodium_mg': facts['sodium_mg'],
        'protein_g': facts['protein_g'],
        'sugars_g': facts['sugars_g'],
        'total_fat_g': facts['total_fat_g'],
        'fiber_g': facts['fiber_g'],
        'saturated_fat_g': facts['saturated_fat_g'],
        'calories': facts['calories'],
        'cuisine_matched': facts['cuisine_matched'],
        'matched_cuisine': '|'.join(facts['matched_cuisine']),
        'key_ingredients': '|'.join(facts['key_ingredients']),
        'dish_type': facts['dish_type'],
        'goal_percentile': facts['goal_percentile'],
        'goal_rank_desc': facts['goal_rank_desc'] or '',
        'goal_rank_phrase': facts['goal_rank_phrase'],
        'secondary_strength': facts['secondary_strength'] or '',
        'caveat': facts['caveat'] or '',
        'quick_to_make': facts['quick_to_make'],
        'matched_goal_tags': '|'.join(facts['matched_goal_tags']),
        'highly_rated': facts['highly_rated'],
        'opening_style': facts['opening_style'],
        'recipe_url': recipe_row.get('recipe_url', ''),
        'ingredients_display': str(recipe_row.get('ingredients_display', [])),
        'explanation': explanation,
        'model_used': model_used,
    }


### Explanation prompt
The prompt supplies only the verified facts. It forbids numbers, absolute nutrient levels, health benefits and nutritional mechanisms.

In [ ]:
def build_grounded_prompt(facts):
    """
    Build the number-free prompt from one recipe's facts. The model must say why
    the recipe was selected (the goal rank) and may name key ingredients and the
    dish type. It may not state numbers, nutrient levels, health benefits or
    nutritional mechanisms
    """
    f = facts

    #  when there are no restrictions, say so and forbid the phrase - Sophie got
    # 'complies with your dietary needs' in 5/5 although she has none.
    if f['dietary_restrictions']:
        dietary_line = f"- The user's dietary needs: {', '.join(f['dietary_restrictions'])} (the recipe complies)"
    else:
        dietary_line = "- The user has NO dietary restrictions. Do not mention dietary needs at all."
    ingredients  = ', '.join(f['key_ingredients']) if f['key_ingredients'] else 'not available'

    if f['matched_cuisine']:
        cuisine_line = f"- Cuisine: this is a {' / '.join(f['matched_cuisine'])} recipe, which matches one of the user's preferred cuisines."
    else:
        cuisine_line = "- Cuisine: this recipe is outside the user's usual cuisine preferences."

    if f['goal_rank_desc']:
        reason_line = (f"- WHY IT WAS SELECTED: compared with the other healthy recipes that fit this user's profile, it is "
                       f"{f['goal_rank_desc']}: one of the {f['goal_rank_phrase']}. Keep the direction word "
                       f"('{f['goal_rank_phrase'].split()[0]}') - never say just 'for sodium content'.")
    else:
        reason_line = f"- WHY IT WAS SELECTED: it passed the screening for {f['nutrient_focus']}."

    # rotate the opening so the set does not read as one template
    openings = {
        0: "Open with the type of dish or a key ingredient (e.g. 'A hearty lentil soup ...').",
        1: "Open with the user's goal (e.g. 'For your weight-loss goal, ...').",
        2: "Open with the reason it was selected, in your own words.",
    }
    opening_rule = openings[f.get('opening_style', 0)]

    optional = []
    if f['matched_goal_tags']:
        optional.append(f"- The recipe author tagged it: {', '.join(f['matched_goal_tags'])}.")
    if f['highly_rated']:
        optional.append("- It is well rated by other home cooks.")
    if f.get('secondary_strength'):
        optional.append(f"- Compared with other recipes that fit the profile, it is also one of the {f['secondary_strength']}.")
    if f.get('quick_to_make'):
        optional.append("- It is quick to prepare.")
    if f.get('caveat'):
        optional.append(f"- One thing to note: compared with other recipes that fit the profile, {f['caveat']}.")
    optional_block = ('\n' + '\n'.join(optional)) if optional else ''

    prompt = f"""You are a knowledgeable and friendly nutrition assistant. Explain to the user why THIS recipe was recommended to them, based STRICTLY on the verified facts below.

VERIFIED FACTS (these are the ONLY facts you may use):
- Recipe name: {f['title']}
- Type of dish: {f['dish_type']}
- Key ingredients: {ingredients}
- The user's health goal: {f['health_goal']} (focus on {f['nutrient_focus']})
{reason_line}
- Overall healthiness: this recipe is {f['balance_desc']} ({f['nutri_gloss']}).
{cuisine_line}
{dietary_line}{optional_block}

STRICT RULES:
1. Use ONLY the facts listed above. Do NOT invent ingredients, nutritional values, or health claims.
2. Do NOT include ANY numbers - no grams, milligrams, calories, percentages or scores. The user sees exact numbers separately.
3. Do NOT state nutrient LEVELS for this recipe (never say "low in sodium", "high in protein", "low fat") and do NOT explain nutritional mechanisms of ingredients (never say "lentils are rich in fibre" or "garlic lowers blood pressure"). The only goal-related statement allowed is the WHY IT WAS SELECTED fact, worded as a ranking among recipes that fit the user's profile.
4. Give the WHY IT WAS SELECTED reason explicitly - this is the core of the explanation. Put it IN YOUR OWN WORDS; do not copy the sentence from the facts verbatim, but do not make it stronger and do not add a nutrient level. Do not present the ranking as the reason the recipe fits a dietary need - those are separate facts.
5. Mention one or two of the key ingredients and/or the type of dish so the user can picture the meal. Do not list all ingredients.
6. State overall healthiness using the exact healthiness phrase and NutriScore grade given, once. Do NOT substitute a stronger term.
7. If a dietary need is listed, you may say the recipe fits it. If the cuisine matches, name the cuisine (e.g. "European"), do not just say "your preferred cuisine". If the cuisine does not match, do not mention cuisine.
8. {opening_rule} Do not start with "I recommend" or "You might enjoy".
9. The facts after the dietary line are optional extras. Use AT MOST TWO of them, and if a "One thing to note" fact is given, include it in one short, neutral clause (it is a relative comparison, not a warning). Never invent an extra of your own.
10. Write 3 to 4 warm, natural sentences addressed to the user. No bullet points. No square brackets. No numbers.
11. Regardless of which opening style you use, mention the health goal by name (e.g. "your Cardiovascular Health goal") at least once somewhere in the text.

Now write the explanation:"""
    return prompt




### Generate the study set
Five explanations per persona, 30 in total. This set is frozen in section 15 and shown to participants.

In [ ]:
def load_explanations():
    """
    Load the pre-generated explanations from disk.
    Returns a DataFrame. Empty if file does not exist yet.
    """
    if not os.path.exists(EXPLANATIONS_FILE):
        print(f"  No explanations file yet at {EXPLANATIONS_FILE}")
        print(f"  Run build_explanation_dataset(TEST_PERSONAS) first.")
        return pd.DataFrame()
    return pd.read_csv(EXPLANATIONS_FILE)


def get_explanations_for_persona(persona_key):
    """
    Return all pre-generated explanations for one persona.
    """
    all_exps = load_explanations()
    if all_exps.empty:
        return pd.DataFrame()
    return all_exps[all_exps['persona_key'] == persona_key].reset_index(drop=True)


def get_explanation(persona_key, recipe_title):
    """
    Return a single pre-generated explanation for a specific
    (persona, recipe) pair, or None if not found.
    """
    all_exps = load_explanations()
    if all_exps.empty:
        return None
    match = all_exps[
        (all_exps['persona_key'] == persona_key) &
        (all_exps['recipe_title'] == recipe_title)
    ]
    if len(match) == 0:
        return None
    return match.iloc[0].to_dict()


In [ ]:
build_explanation_dataset(TEST_PERSONAS, n_per_persona=5)

### Generate the validation set
72 further explanations with the same pipeline, prompt and model. They enlarge the pool of claims for judging and provide the material for the adversarial cases, so the study set is never altered.

In [ ]:
# ── VALIDATION SET GENERATION (separate from the frozen study set) ────────────

VALIDATION_FILE = 'data/explanations_VALIDATION.csv'

def build_validation_set(personas, n_extra_per_persona=12, n_fetch=60):
    """
    For each persona, fetch more recommendations than the study's 5, skip the 5
    already in the frozen study set, and generate explanations for the next
    n_extra_per_persona recipes. Resumable; single pinned model.
    """
    os.makedirs('data', exist_ok=True)

    # Load the study set to know which (persona, recipe) pairs to skip
    frozen = pd.read_csv('data/explanations.csv')
    frozen_keys = set(zip(frozen['persona_key'], frozen['recipe_title']))

    # Get a LARGER recommendation list per persona (diversity-aware, same pipeline)
    all_recs = get_all_recommendations(personas, n=n_fetch)

    for persona_key, persona in personas.items():
        recs = all_recs[persona_key]
        #  the rank is computed against the SAME set apply_goal_floor ranked
        # (filtered + scored + NutriScore-filtered), so every recommended recipe is
        # >= 85th percentile by construction and the 'no rank fact' fallback never fires.
        pool = apply_nutriscore_filter(score_recipes(filter_recipes(clean_recipes, persona), persona), n=5)  

        if os.path.exists(VALIDATION_FILE):
            done = pd.read_csv(VALIDATION_FILE)
            existing_count = len(done[done['persona_key'] == persona_key])
        else:
            done = pd.DataFrame()
            existing_count = 0

        added = 0
        target = n_extra_per_persona - existing_count

        if target <= 0:
            print(f"  [skip persona] {persona['name']} already has {existing_count}/{n_extra_per_persona}")
            continue

        for idx, (_, recipe_row) in enumerate(recs.iterrows()):
            if added >= target:
                break
            title = recipe_row['title']

            # skip if it's in the frozen study set
            if (persona_key, title) in frozen_keys:
                continue

            # skip if already generated in the validation file (resume)
            if os.path.exists(VALIDATION_FILE):
                done = pd.read_csv(VALIDATION_FILE)
                if (persona_key, title) in set(zip(done['persona_key'], done['recipe_title'])):
                    continue
            else:
                done = pd.DataFrame()

            bad = compliance_violations(recipe_row, persona)
            if bad:
                print(f"  [BLOCKED] {persona['name']}: {title} violates {bad} - fix the engine, not the prompt")
                continue
            facts = extract_grounding_facts(recipe_row, persona, pool_df=pool, opening_style=idx)
            prompt = build_grounded_prompt(facts)
            explanation, model_used = generate_single_model(prompt)

            if explanation is None:
                print(f"\n  {GENERATION_MODEL} capped. Progress saved. Re-run to resume.\n")
                return

            new_row = pd.DataFrame([facts_to_row(persona_key, persona, recipe_row, facts, explanation, model_used)])
            combined = pd.concat([done, new_row], ignore_index=True)
            combined.to_csv(VALIDATION_FILE, index=False)
            added += 1
            print(f"  [done] {persona['name']}: {title}")
            time.sleep(2)

    final = pd.read_csv(VALIDATION_FILE)
    print(f"\n  Validation set complete: {len(final)} explanations, models: {final['model_used'].unique().tolist()}")


build_validation_set(TEST_PERSONAS, n_extra_per_persona=12)

## 8. Functional Testing
Black-box checks that the engine respects all hard constraints (allergens, dietary compliance, sodium limit) for the exact recipes participants will see. This corresponds to the testing evaluation method of Hevner et al. (2004).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FUNCTIONAL (BLACK BOX) TEST OF THE RECOMMENDATION ENGINE
# Verifies the engine respects all hard constraints for every persona.
# ══════════════════════════════════════════════════════════════════════════════

def run_functional_tests(personas, n=5):
    """
    Run every persona through the engine and verify all hard constraints.
    Returns a results table and an overall pass/fail.
    """
    results = []
    all_passed = True

    all_recs = get_all_recommendations(personas, n=n)

    for key, persona in personas.items():
        recs = all_recs[key]

        # ── Check 1: Allergen safety (CRITICAL) ───────────────────────────
        allergen_violations = 0
        if persona['allergies']:
            expanded = expand_allergens(persona['allergies'])
            for _, row in recs.iterrows():
                if violates(expanded, row):
                    allergen_violations += 1

        # ── Check 2: Dietary compliance ───────────────────────────────────
        dietary_violations = 0
        for restriction in persona['dietary_restrictions']:
            if restriction in DIETARY_VIOLATIONS:
                violations = DIETARY_VIOLATIONS[restriction]
                for _, row in recs.iterrows():
                    if violates(violations, row):
                        dietary_violations += 1

        # ── Check 3: Dislike avoidance ────────────────────────────────────
        dislike_violations = 0
        for _, row in recs.iterrows():
            if persona['disliked_ingredients'] and \
               violates(persona['disliked_ingredients'], row):
                dislike_violations += 1

        # ── Check 3b : non-meal titles / calorie floor ─────────────────
        nonmeal_violations = sum(
            1 for _, row in recs.iterrows()
            if NON_MEAL_TITLE_RE.search(str(row['title'])) or row['calories [cal]'] < MIN_CALORIES
        )

        # ── Check 4: Sodium safety floor ──────────────────────────────────
        sodium_violations = sum(
            1 for _, row in recs.iterrows() if row['sodium [mg]'] > 500
        )

        # ── Check 5: Cuisine match (informational) ────────────────────────
        cuisine_matches = sum(
            1 for _, row in recs.iterrows()
            if any(c in parse_tags(row['tags']) for c in persona['cuisine_preferences'])
        )

        # ── Check 6: Diversity (informational) ────────────────────────────
        categories = [get_recipe_category(row['tags']) for _, row in recs.iterrows()]
        max_same_category = max(
            (categories.count(c) for c in set(categories)), default=0
        )
        diversity_ok = max_same_category <= 2

        # ── Determine pass/fail for this persona ──────────────────────────
        # Critical and high checks must be zero
        persona_passed = (
            allergen_violations == 0 and
            dietary_violations == 0 and
            dislike_violations == 0 and
            sodium_violations == 0 and
            nonmeal_violations == 0
        )
        if not persona_passed:
            all_passed = False

        results.append({
            'persona': persona['name'],
            'recipes': len(recs),
            'allergen_viol': allergen_violations,
            'dietary_viol': dietary_violations,
            'dislike_viol': dislike_violations,
            'sodium_viol': sodium_violations,
            'nonmeal_viol': nonmeal_violations,
            'cuisine_match': f"{cuisine_matches}/{len(recs)}",
            'diversity_ok': 'Yes' if diversity_ok else 'No',
            'passed': 'PASS' if persona_passed else 'FAIL',
        })

    return pd.DataFrame(results), all_passed


# ── Run the tests ─────────────────────────────────────────────────────────────
test_results, overall = run_functional_tests(TEST_PERSONAS)

print("="*78)
print("FUNCTIONAL TEST RESULTS - Recommendation Engine")
print("="*78)
print(test_results.to_string(index=False))
print("="*78)
print(f"OVERALL: {'ALL TESTS PASSED' if overall else 'SOME TESTS FAILED - INVESTIGATE'}")
print("="*78)

# Save for Chapter 4
test_results.to_csv('data/functional_test_results.csv', index=False)
print("\nSaved to data/functional_test_results.csv")

## 9. Judge Setup
The two judge models, Qwen3.6-35B-A3B and GPT-OSS 120B, both from model families different from the Gemma generator.

In [ ]:
import os, time, json, re
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

load_dotenv()
lehmus_key = os.getenv('LEHMUS_API_KEY')
lehmus_url = os.getenv('LEHMUS_BASE_URL')
lehmus_client = OpenAI(api_key=lehmus_key, base_url=lehmus_url)

JUDGE_MODELS = {
    'qwen':    'ctraxdkjuwfrlivzrhgt',   # Qwen3.6-35B-A3B
    'gpt-oss': 'azsydsttjnlbfjbgqnwd',   # GPT OSS 120B
}
print("Lehmus client ready. Judges:", list(JUDGE_MODELS))

## 10. Per-Claim Judge
One judge prompt is used throughout. It labels each of the five claims (ingredients and dish, goal ranking, healthiness, cuisine and dietary compliance) as Supported or Unsupported against its grounding fact. It also rules out nutrient numbers outside the recipe title, absolute nutrient levels and added health claims. The verdict for a whole explanation is rolled up from these labels in Section 12. Like the generator prompt, the wording should not be edited.

In [ ]:
NUTRI_DESCRIPTOR = {
    'A': 'a nutritionally strong choice',
    'B': 'a healthy choice',
    'C': 'a moderate choice',
    'D': 'a lower-scoring choice that is reasonable in moderation',
    'E': 'an indulgent choice, best enjoyed occasionally',
}

def build_perclaim_judge_prompt(row, persona):
    ings  = row['key_ingredients'] if pd.notna(row['key_ingredients']) else ''
    dish  = row['dish_type'] if pd.notna(row['dish_type']) else 'dish'
    grade = row['nutri_grade']
    rank_desc   = row['goal_rank_desc'] if pd.notna(row['goal_rank_desc']) else ''
    rank_phrase = row['goal_rank_phrase'] if pd.notna(row['goal_rank_phrase']) else ''
    matched_cuisine = row['matched_cuisine'] if pd.notna(row['matched_cuisine']) else ''

    core = [
        f'- "ingredients_dish": GROUND TRUTH: the recipe is a {dish}; key ingredients: {ings}. '
        f'Supported if the explanation names only ingredients/dish from this list (a subset is fine); '
        f'Unsupported if it names an ingredient or dish type NOT in the ground truth.',

        (f'- "goal_rank": GROUND TRUTH: the recipe is {rank_desc}, i.e. one of the {rank_phrase} among recipes '
         f'fitting the user profile. Supported if the explanation states this ranking with the CORRECT direction '
         f'("{rank_phrase}"); Unsupported if it flips direction, overstates the tier, or states an absolute '
         f'nutrient level (e.g. "low in sodium") instead of a ranking.') if rank_desc
        else '- "goal_rank": GROUND TRUTH: no ranking claim is licensed for this recipe. Supported if the explanation '
             'makes no comparative ranking claim; Unsupported if it invents one.',

        f'- "healthiness": GROUND TRUTH: NutriScore {grade}. Supported if the explanation states grade {grade} with a '
        f'matching descriptor and no stronger term; Unsupported otherwise.',

        (f'- "cuisine": GROUND TRUTH: matches cuisine(s): {matched_cuisine}. Supported if it names one of these; '
         f'Unsupported if it claims a different cuisine.') if matched_cuisine
        else '- "cuisine": GROUND TRUTH: recipe does NOT match the user\'s cuisines. Supported if the explanation makes '
             'no cuisine-match claim; Unsupported if it claims a match.',

        (f'- "dietary": GROUND TRUTH: dietary needs = {", ".join(persona["dietary_restrictions"])} (the recipe complies). '
         f'Supported if the compliance statement matches; Unsupported if it claims compliance with something violated.')
        if persona['dietary_restrictions']
        else '- "dietary": GROUND TRUTH: the user has NO dietary restrictions. Supported if the explanation makes no '
             'dietary-needs claim; Unsupported if it claims the recipe fits a dietary need.',
    ]

    return f"""You are a strict fact-checker. Judge EACH of the five claims below as "Supported" or "Unsupported"
against its GROUND TRUTH. The explanation must be number-free (except numbers inside the recipe name).

ABSOLUTE RULES:
- A nutrient NUMBER anywhere (except in the recipe title) -> the claim it appears in is Unsupported.
- An absolute nutrient LEVEL ("low in sodium", "high in protein") -> Unsupported; only a RANKING among
  profile-fitting recipes is allowed for the goal.
- An added health/medical benefit or nutritional mechanism ("garlic lowers blood pressure") -> Unsupported.

CLAIMS:
{chr(10).join(core)}

EXPLANATION:
"{row['explanation']}"

Respond ONLY with JSON in exactly this form:
{{"ingredients_dish": "Supported", "goal_rank": "Supported", "healthiness": "Supported", "cuisine": "Supported", "dietary": "Supported"}}"""


def judge_claims(row, persona, model, max_retries=3):
    prompt = build_perclaim_judge_prompt(row, persona)
    for attempt in range(max_retries):
        try:
            resp = lehmus_client.chat.completions.create(
                model=model, messages=[{"role":"user","content":prompt}], temperature=0)
            raw = resp.choices[0].message.content.strip()
            raw = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL)
            raw = re.sub(r'```json|```', '', raw)
            for cand in reversed(re.findall(r'\{[^{}]*\}', raw, re.DOTALL)):
                try:
                    result = json.loads(cand)
                    if isinstance(result, dict) and result:
                        return {k.lower(): v for k, v in result.items()}
                except json.JSONDecodeError:
                    continue
            if attempt < max_retries-1: time.sleep(2); continue
            return {}
        except Exception:
            if attempt < max_retries-1: time.sleep(3); continue
            return {}
    return {}
print("Per-claim judge ready.")

## 11. Claim Decomposition and Claim-Level Judging (Natural Cases)
Split all 102 explanations (30 study and 72 validation) into their claims and run both judges on every claim.

In [ ]:
import time as time_module

study = pd.read_csv('data/explanations.csv');            study['set'] = 'study'
valid = pd.read_csv('data/explanations_VALIDATION.csv'); valid['set'] = 'validation'
pool  = pd.concat([study, valid], ignore_index=True)
print(f"Annotation pool: {len(pool)} explanations")

CORE_CLAIMS = ['ingredients_dish', 'goal_rank', 'healthiness', 'cuisine', 'dietary']

def decompose_row(r):
    """One row per core claim. The 'dietary' claim is omitted for personas with
    no restrictions (there is nothing to verify)."""
    persona = TEST_PERSONAS[r['persona_key']]
    expl_id = f"{r['persona_key']}__{r['recipe_title']}"[:80]
    # ground-truth summary shown to annotators, per claim
    mc = r['matched_cuisine'] if pd.notna(r['matched_cuisine']) else ''
    facts = {
        'ingredients_dish': f"Dish: {r['dish_type']}. Key ingredients: {r['key_ingredients']}",
        'goal_rank': f"Goal: {r['health_goal']}. Ranking: {r['goal_rank_desc']} ({r['goal_rank_phrase']}) among profile-fitting recipes",
        'healthiness': f"NutriScore {r['nutri_grade']}",
        'cuisine': f"Matched cuisine: {mc}" if mc else "No cuisine match",
        'dietary': f"Dietary needs: {', '.join(persona['dietary_restrictions']) if persona['dietary_restrictions'] else 'none'}",
    }
    rows = []
    for ct in CORE_CLAIMS:
        if ct == 'dietary' and not persona['dietary_restrictions']:
            continue  # nothing to verify for no-restriction personas
        rows.append({
            'claim_id': f"{expl_id}__{ct}",
            'explanation_id': expl_id,
            'persona': persona['name'], 'persona_key': r['persona_key'],
            'health_goal': r['health_goal'], 'recipe_title': r['recipe_title'],
            'claim_type': ct, 'grounding_fact': facts[ct],
            'full_explanation': r['explanation'], 'set': r['set'],
        })
    return rows

claim_rows = []
for _, r in pool.iterrows():
    claim_rows.extend(decompose_row(r))
claims_df = pd.DataFrame(claim_rows)
print(f"Total core claims: {len(claims_df)}")
print(claims_df['claim_type'].value_counts())

# ── Run both judges (per-explanation call returns all 5 verdicts at once) ──────
CHECKPOINT = 'data/claims_with_judges_PARTIAL.csv'
total = len(pool)
for short, model in JUDGE_MODELS.items():
    print(f"\n{'='*60}\nJudging with {short} ({total} explanations)\n{'='*60}")
    verdicts, failures, start = {}, 0, time_module.time()
    for i, (_, r) in enumerate(pool.iterrows(), start=1):
        persona = TEST_PERSONAS[r['persona_key']]
        expl_id = f"{r['persona_key']}__{r['recipe_title']}"[:80]
        result = judge_claims(r, persona, model)
        if not result:
            failures += 1
            print(f"  [{i}/{total}] FAILED — {r['persona_name']} / {r['recipe_title'][:35]}")
        else:
            for ct in CORE_CLAIMS:
                if ct in result:
                    verdicts[f"{expl_id}__{ct}"] = result[ct]
        if i % 5 == 0 or i == total:
            el = time_module.time() - start
            print(f"  [{i}/{total}] {el:.0f}s, ~{el/i*(total-i):.0f}s left, {failures} failures")
        if i % 10 == 0:
            claims_df[f'judge_{short}'] = claims_df['claim_id'].map(verdicts)
            claims_df.to_csv(CHECKPOINT, index=False)
        time_module.sleep(0.4)
    claims_df[f'judge_{short}'] = claims_df['claim_id'].map(verdicts)
    print(f"  {short} done: {failures} failures")

claims_df.to_csv('data/claims_with_judges.csv', index=False)
print("\nSaved data/claims_with_judges.csv")
print("NaN check — qwen:", claims_df['judge_qwen'].isna().sum(),
      "| gpt-oss:", claims_df['judge_gpt-oss'].isna().sum())

In [ ]:
# Remove tracker

import os
if os.path.exists('data/claims_with_judges_PARTIAL.csv'):
    os.remove('data/claims_with_judges_PARTIAL.csv')

In [ ]:
import pandas as pd

c = pd.read_csv('data/claims_with_judges.csv')
print("qwen:\n", c['judge_qwen'].value_counts())
print("gpt-oss:\n", c['judge_gpt-oss'].value_counts())
# any claims either judge flagged Unsupported?
flagged = c[(c['judge_qwen']=='Unsupported') | (c['judge_gpt-oss']=='Unsupported')]
print(f"\nFlagged claims: {len(flagged)}")
print(flagged[['persona','claim_type','recipe_title','judge_qwen','judge_gpt-oss']].to_string() if len(flagged) else "none")

In [ ]:
# More details on disagreements

c = pd.read_csv('data/claims_with_judges.csv')
flagged = c[(c['judge_qwen']=='Unsupported') | (c['judge_gpt-oss']=='Unsupported')]
for _, r in flagged.iterrows():
    print(f"[{r['claim_type']}] {r['persona']} / {r['recipe_title']}")
    print(f"  FACT: {r['grounding_fact']}")
    print(f"  EXPL: {r['full_explanation'][:180]}")
    print(f"  qwen={r['judge_qwen']} gpt-oss={r['judge_gpt-oss']}")
    print()

## 12. Grounding Verification
Grounding PASS or FAIL for each explanation, rolled up from the claim-level verdicts.

In [ ]:
# ── GROUNDING VERIFICATION (rolled up from per-claim results) ──────────────────
# An explanation PASSES grounding if ALL its claims are Supported by a judge.
claims = pd.read_csv('data/claims_with_judges.csv')

def explanation_verdicts(claims_df, judge_col):
    g = claims_df.groupby('explanation_id')[judge_col].apply(
        lambda s: 'PASS' if (s == 'Supported').all() else 'FAIL')
    return g

# map each explanation_id to its set (study / validation)
set_map = claims.drop_duplicates('explanation_id').set_index('explanation_id')['set']

expl_v = pd.DataFrame({
    'qwen': explanation_verdicts(claims, 'judge_qwen'),
    'gpt-oss': explanation_verdicts(claims, 'judge_gpt-oss'),
}).reset_index()
expl_v['set'] = expl_v['explanation_id'].map(set_map)
expl_v.to_csv('data/grounding_verdicts.csv', index=False)

# Report overall AND study-only (the 30 participants see) AND validation-only
for scope, sub in [('ALL 102', expl_v),
                   ('STUDY 30 (participant-facing)', expl_v[expl_v['set']=='study']),
                   ('VALIDATION 72', expl_v[expl_v['set']=='validation'])]:
    print(f"\n{scope}:")
    for jc in ['qwen', 'gpt-oss']:
        print(f"  {jc}: {(sub[jc]=='PASS').sum()}/{len(sub)} explanations PASS grounding")

## 13. Inter-Judge Agreement
Cohen's kappa and raw agreement between the two judges on the claim-level verdicts.

In [ ]:
from sklearn.metrics import cohen_kappa_score
claims = pd.read_csv('data/claims_with_judges.csv')
c = claims.dropna(subset=['judge_qwen','judge_gpt-oss'])
c = c[c['judge_qwen'].isin(['Supported','Unsupported']) & c['judge_gpt-oss'].isin(['Supported','Unsupported'])]

kappa = cohen_kappa_score(c['judge_qwen'], c['judge_gpt-oss'])
agree = (c['judge_qwen']==c['judge_gpt-oss']).mean()
print(f"Per-claim inter-judge Cohen's kappa: {kappa:.3f}")
print(f"Per-claim raw agreement: {agree:.1%}")
print(f"Qwen Supported: {(c['judge_qwen']=='Supported').sum()}/{len(c)}")
print(f"gpt-oss Supported: {(c['judge_gpt-oss']=='Supported').sum()}/{len(c)}")

## 14. Judge Validation: Adversarial Cases
Natural explanations are almost all faithful, so known errors are planted to measure whether the judges detect them. There are eight error types (seven obvious and one subtle) with six cases each, 48 in total. The adversarial cases are drawn from the validation set, so the study set stays untouched. Natural and adversarial cases are reported separately.

In [ ]:
FAR_GRADE = {'A':'E','B':'E','C':'A','D':'A','E':'A'}
ADJACENT_GRADE = {'A':'B','B':'C','C':'B','D':'C','E':'D'}
ANIMAL = ['clam','mussel','oyster','shrimp','prawn','crab','lobster','scallop','fish',
          'chicken','beef','pork','lamb','bacon','ham','turkey','anchov','salmon','tuna',
          'milk','cheese','egg','butter','cream','yogurt','honey','meat','veal','duck','squid','octopus']
FAKE_INGS = ['salmon','peanuts','blue cheese','chocolate','bacon','shrimp']

def _non_vegan(ing): return any(a in str(ing).lower() for a in ANIMAL)

def _adv_case(r, expl, etype, tier, target, label, justification):
    return {'persona_key':r['persona_key'],'persona_name':r['persona_name'],
            'health_goal':r['health_goal'],'recipe_title':r['recipe_title'],
            'nutri_grade':r['nutri_grade'],'cuisine_matched':r['cuisine_matched'],
            'matched_cuisine':r.get('matched_cuisine',''),'key_ingredients':r.get('key_ingredients',''),
            'dish_type':r.get('dish_type',''),'goal_rank_desc':r.get('goal_rank_desc',''),
            'goal_rank_phrase':r.get('goal_rank_phrase',''),'ingredients_display':r.get('ingredients_display',''),
            'original_explanation':r['explanation'],'explanation':expl,
            'error_type':etype,'tier':tier,'target_claim':target,
            'known_label':label,'justification':justification,'set':'adversarial'}

def make_adversarial(pool, personas, n_each=6, seed=42):
    rng = pd.Series(range(len(pool))).sample(frac=1, random_state=seed).tolist()
    used = set()
    def take(n, cond=None):
        out=[]
        for i in rng:
            if i in used: continue
            r = pool.iloc[i]
            if cond and not cond(r): continue
            out.append(i); used.add(i)
            if len(out)==n: break
        return out
    cases=[]

    # healthiness: wrong grade (far)
    for i in take(n_each):
        r=pool.iloc[i]; g=r['nutri_grade']; w=FAR_GRADE[g]
        cases.append(_adv_case(r, re.sub(r'NutriScore [A-E]', f'NutriScore {w}', r['explanation']),
            'wrong_grade','obvious','healthiness','Unsupported', f"True grade {g}; states {w}."))
    # healthiness: injected number
    for i in take(n_each):
        r=pool.iloc[i]
        cases.append(_adv_case(r, re.sub(r'\.', ', providing 27 grams of protein per serving.', r['explanation'], count=1),
            'injected_number','obvious','healthiness','Unsupported', "Adds a numeric nutrient value."))
    # healthiness: fabricated health claim
    for i in take(n_each):
        r=pool.iloc[i]
        cases.append(_adv_case(r, r['explanation']+' It is also excellent for boosting your heart health and immunity.',
            'fabricated_health','obvious','healthiness','Unsupported', "Adds an unsupported medical claim."))
    # cuisine: false cuisine
    for i in take(n_each):
        r=pool.iloc[i]
        expl = r['explanation']+' It also matches your preferred Japanese cuisine.'
        cases.append(_adv_case(r, expl, 'false_cuisine','obvious','cuisine','Unsupported',
            "Claims a Japanese match no persona has."))
    # dietary: false vegan claim on a recipe with animal ingredients
    for i in take(n_each, cond=lambda r: _non_vegan(r.get('ingredients_display',''))):
        r=pool.iloc[i]
        cases.append(_adv_case(r, r['explanation']+' This dish is also fully vegan-friendly.',
            'false_dietary','obvious','dietary','Unsupported', "Animal ingredients; 'vegan' is false."))
    # NEW - goal_rank: flip the ranking direction
    def _flip(e):
        for a,b in [('lowest in','highest in'),('highest in','lowest in'),('lower in','higher in')]:
            if a in e: return e.replace(a,b,1)
        return None
    for i in take(n_each, cond=lambda r: _flip(r['explanation']) is not None):
        r=pool.iloc[i]
        cases.append(_adv_case(r, _flip(r['explanation']),
            'false_goal_rank','obvious','goal_rank','Unsupported',
            "Ranking direction flipped (contradicts the computed rank)."))
    # NEW - ingredients_dish: inject an ingredient not in the recipe
    for i in take(n_each):
        r=pool.iloc[i]
        real = str(r.get('ingredients_display','')).lower()
        fake = next((f for f in FAKE_INGS if f not in real), 'salmon')
        cases.append(_adv_case(r, r['explanation'].rstrip('.')+f", and it features {fake}.",
            'wrong_ingredient','obvious','ingredients_dish','Unsupported',
            f"Injected '{fake}', not in the recipe."))
    # subtle: inflated descriptor
    for i in take(n_each):
        r=pool.iloc[i]; g=r['nutri_grade']; correct=NUTRI_DESCRIPTOR[g]
        if correct not in r['explanation']: continue
        cases.append(_adv_case(r, r['explanation'].replace(correct,'an exceptionally healthy, top-tier choice'),
            'inflated_descriptor','subtle','healthiness','Unsupported', f"Grade {g} inflated."))

    return pd.DataFrame(cases)

print("Adversarial toolkit loaded.")

In [ ]:
# Adversarial cases

valid = pd.read_csv('data/explanations_VALIDATION.csv')
adversarial = make_adversarial(valid, TEST_PERSONAS, n_each=6)
adversarial.to_csv('data/adversarial_cases.csv', index=False)
print(f"Generated {len(adversarial)} adversarial cases")
print(adversarial.groupby(['error_type','target_claim']).size())

# REVIEW every SUBTLE case - confirm each 'Unsupported' label is defensible
print("\n" + "="*70)
print("REVIEW SUBTLE CASES:")
for _, r in adversarial[adversarial['tier']=='subtle'].iterrows():
    print(f"\n[{r['error_type']}] {r['persona_name']} / {r['recipe_title']}")
    print(f"  ORIGINAL:  {r['original_explanation']}")
    print(f"  MODIFIED:  {r['explanation']}")
    print(f"  JUSTIFICATION: {r['justification']}")

In [ ]:
# Run both judges on the adversarial set; record the verdict on the TARGET claim

import time as tm

# Ensure we have a unique ID for each adversarial case. 
# If you didn't add an 'id' column in make_adversarial, create one now:
adversarial_reset = adversarial.reset_index(drop=True)
adversarial_reset['case_id'] = [f"ADV-{i+1}" for i in range(len(adversarial_reset))]

adv_case_results = []
total = len(adversarial_reset)
start = tm.time()
failures = 0

print(f"Judging {total} adversarial cases...\n")

for i, (_, r) in enumerate(adversarial_reset.iterrows(), start=1):
    persona = TEST_PERSONAS[r['persona_key']]
    tgt = r['target_claim']
    
    # Initialize a record for this specific case
    record = {
        'case_id': r['case_id'],
        'persona_key': r['persona_key'],
        'persona_name': r['persona_name'],
        'recipe_title': r['recipe_title'],
        'error_type': r['error_type'],
        'tier': r['tier'],
        'target_claim': tgt,
        'known_label': r['known_label'],
        'justification': r['justification'],
        'original_explanation': r['original_explanation'],
        'modified_explanation': r['explanation'],
        'grounding_fact_for_claim': r.get('justification', 'N/A')
    }

    # Run judges
    for short, model in JUDGE_MODELS.items():
        v = judge_claims(r, persona, model)
        verdict = v.get(tgt, 'MISSING')
        
        if not v or verdict == 'MISSING':
            failures += 1
            record[f'judge_{short}'] = 'ERROR'
        else:
            record[f'judge_{short}'] = verdict
            
        # Determine if caught (only if we have a valid verdict)
        if verdict != 'MISSING' and verdict != 'ERROR':
            record[f'caught_{short}'] = (verdict == 'Unsupported')
        else:
            record[f'caught_{short}'] = None

    adv_case_results.append(record)
    
    # Heartbeat
    if i % 5 == 0 or i == total:
        el = tm.time() - start
        print(f"  [{i}/{total}] {el:.0f}s elapsed, {failures} errors so far")
    tm.sleep(0.4)

# Create the Per-Case DataFrame
adv_case_df = pd.DataFrame(adv_case_results)

# Save the detailed per-case results
adv_case_df.to_csv('data/adversarial_judged_PER_CASE.csv', index=False)

# ALSO save the simple summary for quick charts (optional but good)
summary_rows = []
for _, r in adv_case_df.iterrows():
    for judge in ['qwen', 'gpt-oss']:
        summary_rows.append({
            'error_type': r['error_type'],
            'judge': judge,
            'verdict': r[f'judge_{judge}'],
            'known': r['known_label'],
            'caught': r[f'caught_{judge}']
        })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('data/adversarial_judged_SUMMARY.csv', index=False)

print(f"\nSaved data/adversarial_judged_PER_CASE.csv ({len(adv_case_df)} rows)")
print(f"Saved data/adversarial_judged_SUMMARY.csv ({len(summary_df)} rows)")

## 15. Freeze the Study Set
Copy the study explanations to `data/explanations_FROZEN.csv`, the file the Streamlit app reads.

In [ ]:
import shutil
shutil.copy('data/explanations.csv', 'data/explanations_FROZEN.csv')
print("Frozen. This is the file the Streamlit app will read.")
print(pd.read_csv('data/explanations_FROZEN.csv')['model_used'].value_counts())

## 16. Annotation
Build the blind annotation workbook of 152 claims, record the development findings, and run a third model on the judges' disagreements.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BUILD BLIND ANNOTATION WORKBOOK — FIXED VERSION
# Merges adversarial judge verdicts from adversarial_judged_PER_CASE.csv
# into the annotation key so annotators' adjudication is grounded in
# actual judge behavior for adversarial cases.
# ══════════════════════════════════════════════════════════════════════════════

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter
import os

# ── Load all source data ──────────────────────────────────────────────────────
claims = pd.read_csv('data/claims_with_judges.csv')

# Natural explanations pool (study + validation)
study = pd.read_csv('data/explanations.csv')
valid = pd.read_csv('data/explanations_VALIDATION.csv')
natural_pool = pd.concat([study, valid], ignore_index=True)
natural_pool['set'] = ['study'] * len(study) + ['validation'] * len(valid)

# Adversarial cases with their modified explanations
adv = pd.read_csv('data/adversarial_cases.csv')

# Adversarial judge results — this is where the LLM verdicts live
adv_judged = pd.read_csv('data/adversarial_judged_PER_CASE.csv')

print(f"Natural claims: {len(claims)}")
print(f"Adversarial cases: {len(adv)}")
print(f"Adversarial judged rows: {len(adv_judged)}")

# ── 1. Select Natural Claims ──────────────────────────────────────────────────
# A. All judge-disagreement claims
disagree = claims[claims['judge_qwen'] != claims['judge_gpt-oss']].copy()
disagree['source'] = 'natural_disagree'
disagree['known_label'] = 'Disagreement'

# B. Stratified sample of agreed-Supported naturals
agreed = claims[(claims['judge_qwen'] == 'Supported') & (claims['judge_gpt-oss'] == 'Supported')].copy()
groups = agreed.groupby(['persona', 'claim_type'])
per_group = max(1, round(90 / len(groups)))
parts = [g.sample(min(len(g), per_group), random_state=42) for _, g in groups]
strat = pd.concat(parts, ignore_index=True)
if len(strat) > 90:
    strat = strat.sample(90, random_state=42)
strat['source'] = 'natural_agreed'
strat['known_label'] = 'Supported'


adv_lookup = adv_judged.set_index(['persona_key', 'recipe_title', 'error_type'])

adv_rows = []
TEST_PERSONAS_LOCAL = {} 


try:
    TEST_PERSONAS_LOCAL = TEST_PERSONAS
except NameError:
    # Fallback: reconstruct minimal persona info from adversarial_cases.csv
    for _, r in adv.iterrows():
        pk = r['persona_key']
        if pk not in TEST_PERSONAS_LOCAL:
            TEST_PERSONAS_LOCAL[pk] = {
                'name': r['persona_name'],
                'dietary_restrictions': [],
            }

# Get dietary restrictions from the original natural pool if available
for pk in TEST_PERSONAS_LOCAL:
    mask = natural_pool['persona_key'] == pk
    if mask.any():
        # Dietary restrictions are not in the CSV directly, but we can
        # infer from the claims. For annotation purposes, the key column is what matters.
        pass

for _, r in adv.iterrows():
    persona = TEST_PERSONAS_LOCAL.get(r['persona_key'], {'name': r['persona_name'], 'dietary_restrictions': []})
    ct = r['target_claim']
    pk = r['persona_key']
    title = r['recipe_title']
    etype = r['error_type']

    # Look up the judge verdicts for this specific adversarial case
    try:
        judged_row = adv_lookup.loc[(pk, title, etype)]
        judge_qwen = judged_row['judge_qwen'] if pd.notna(judged_row['judge_qwen']) else 'ERROR'
        judge_gpt_oss = judged_row['judge_gpt-oss'] if pd.notna(judged_row['judge_gpt-oss']) else 'ERROR'
        caught_qwen = judged_row.get('caught_qwen', None)
        caught_gpt = judged_row.get('caught_gpt-oss', None)
    except KeyError:
        judge_qwen = 'NOT_JUDGED'
        judge_gpt_oss = 'NOT_JUDGED'
        caught_qwen = None
        caught_gpt = None

    # Construct the grounding fact string for the annotator
    mc = r.get('matched_cuisine', '')
    fact = {
        'ingredients_dish': f"Dish: {r.get('dish_type', '')}. Key ingredients: {r.get('key_ingredients', '')}",
        'goal_rank': f"Goal: {r['health_goal']}. Ranking: {r.get('goal_rank_desc', '')} ({r.get('goal_rank_phrase', '')}) among profile-fitting recipes",
        'healthiness': f"NutriScore {r['nutri_grade']}",
        'cuisine': f"Matched cuisine: {mc}" if mc else "No cuisine match",
        'dietary': f"Dietary needs: {', '.join(persona.get('dietary_restrictions', [])) if persona.get('dietary_restrictions') else 'none'}",
    }.get(ct, '')

    adv_rows.append({
        'claim_id': f"adv__{pk}__{title}__{ct}"[:90],
        'persona': r['persona_name'],
        'persona_key': pk,
        'health_goal': r['health_goal'],
        'recipe_title': title,
        'claim_type': ct,
        'grounding_fact': fact,
        'full_explanation': r['explanation'],  # the MODIFIED explanation
        'source': 'adversarial',
        'judge_qwen': judge_qwen,
        'judge_gpt-oss': judge_gpt_oss,
        'known_label': r['known_label'],
        'error_type': etype,
        'tier': r['tier'],
        'caught_by_qwen': caught_qwen,
        'caught_by_gpt': caught_gpt,
    })

adv_df = pd.DataFrame(adv_rows)

print(f"\nAdversarial rows prepared: {len(adv_df)}")
print("Judge verdict distribution in adversarial set:")
print(f"  qwen:    {adv_df['judge_qwen'].value_counts().to_dict()}")
print(f"  gpt-oss: {adv_df['judge_gpt-oss'].value_counts().to_dict()}")

# ── 3. Combine, Shuffle, and Number ───────────────────────────────────────────
cols = ['claim_id', 'persona', 'persona_key', 'health_goal', 'recipe_title', 'claim_type',
        'grounding_fact', 'full_explanation', 'source', 'judge_qwen', 'judge_gpt-oss', 'known_label']

# Ensure all columns exist in each dataframe
for col in cols:
    if col not in disagree.columns:
        disagree[col] = ''
    if col not in strat.columns:
        strat[col] = ''
    if col not in adv_df.columns:
        adv_df[col] = ''

combined = pd.concat([disagree[cols], strat[cols], adv_df[cols]], ignore_index=True)
combined = combined.sample(frac=1, random_state=7).reset_index(drop=True)
combined.insert(0, 'row_id', range(1, len(combined) + 1))

print(f"\nTotal annotation rows: {len(combined)}")
print(combined['source'].value_counts())

# ── 4. Save PRIVATE Key ───────────────────────────────────────────────────────
# This file is for the researcher. It maps row_id to source, judge verdicts,
# and known label.
key_cols = ['row_id', 'claim_id', 'source', 'judge_qwen', 'judge_gpt-oss', 'known_label']
# Add adversarial-specific columns if present
if 'error_type' in combined.columns:
    key_cols += ['error_type']
if 'tier' in combined.columns:
    key_cols += ['tier']
if 'caught_by_qwen' in combined.columns:
    key_cols += ['caught_by_qwen', 'caught_by_gpt']

# Fill NaN with empty string for CSV cleanliness
key_df = combined[key_cols].copy()
for c in key_df.columns:
    key_df[c] = key_df[c].fillna('')

key_df.to_csv('data/annotation_KEY.csv', index=False)
print(f"\nSaved data/annotation_KEY.csv ({len(key_df)} rows)")
print("Private key columns:", key_df.columns.tolist())

# ── 5. Build the Blind .xlsx ──────────────────────────────────────────────────
# Annotators see: Row, Persona, Goal, Recipe, Claim Type, Grounding Fact, Explanation
# Annotators DO NOT see: Source, Judge Verdicts, Known Label, Error Type

wb = Workbook()
ws = wb.active
ws.title = "Annotation"
F = 'Arial'

intro = [
    "EXPLANATION GROUNDING ANNOTATION",
    "For each row, decide whether the CLAIM is SUPPORTED by the GROUNDING FACT shown.",
    "Supported   = the claim follows from the fact without adding, exaggerating, or contradicting it.",
    "Unsupported = the claim contradicts the fact, overstates it, adds ungrounded information, or contains a nutrient number.",
    "The 'This claim' column tells you which part of the explanation to judge.",
]
for i, t in enumerate(intro, start=1):
    ws.cell(row=i, column=1, value=t).font = Font(name=F, size=11, bold=(i == 1))

h = len(intro) + 2
headers = ['Row', 'Persona', 'Health Goal', 'Recipe', 'This claim', 'Grounding Fact',
           'Full Explanation',
           'Annotator 1 (Supported/Unsupported)',
           'Annotator 2 (Supported/Unsupported)',
           'Notes']

CLAIM_LABEL = {
    'cuisine': 'the CUISINE-match statement',
    'goal_rank': 'the GOAL-RANKING statement',
    'healthiness': 'the HEALTHINESS (NutriScore) statement',
    'ingredients_dish': 'the INGREDIENTS/DISH statement',
    'dietary': 'the DIETARY-compliance statement',
}

for col, hd in enumerate(headers, start=1):
    cc = ws.cell(row=h, column=col, value=hd)
    cc.font = Font(name=F, size=11, bold=True, color='FFFFFF')
    cc.fill = PatternFill('solid', fgColor='4472C4')
    cc.alignment = Alignment(wrap_text=True, vertical='top')

for idx, (_, r) in enumerate(combined.iterrows()):
    rr = h + 1 + idx
    vals = [
        r['row_id'],
        r['persona'],
        r['health_goal'],
        r['recipe_title'],
        CLAIM_LABEL.get(r['claim_type'], r['claim_type']),
        r['grounding_fact'],
        r['full_explanation'],
        '',
        '',
        '',
    ]
    for col, v in enumerate(vals, start=1):
        cell = ws.cell(row=rr, column=col, value=v)
        cell.font = Font(name=F, size=10)
        cell.alignment = Alignment(wrap_text=True, vertical='top')
    for col in (8, 9):
        ws.cell(row=rr, column=col).fill = PatternFill('solid', fgColor='FFF2CC')

dv = DataValidation(type="list", formula1='"Supported,Unsupported"', allow_blank=True)
ws.add_data_validation(dv)
dv.add(f"H{h + 1}:I{h + len(combined)}")

for col, w in enumerate([5, 10, 20, 26, 24, 42, 58, 20, 20, 22], start=1):
    ws.column_dimensions[get_column_letter(col)].width = w

ws.freeze_panes = f"A{h + 1}"
wb.save('data/annotation_workbook.xlsx')

print(f"Saved data/annotation_workbook.xlsx ({len(combined)} rows — send to annotators)")
print(f"Saved data/annotation_KEY.csv ({len(key_df)} rows — KEEP PRIVATE)")

# ── Verification ──────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("VERIFICATION: Adversarial rows in annotation key")
print("=" * 70)
adv_in_key = key_df[key_df['source'] == 'adversarial']
print(f"Adversarial rows: {len(adv_in_key)}")
if len(adv_in_key) > 0:
    print(adv_in_key[['row_id', 'claim_id', 'judge_qwen', 'judge_gpt-oss', 'known_label']].head(10).to_string(index=False))
    print(f"\nJudge verdict coverage:")
    print(f"  qwen:    {adv_in_key['judge_qwen'].value_counts().to_dict()}")
    print(f"  gpt-oss: {adv_in_key['judge_gpt-oss'].value_counts().to_dict()}")
else:
    print("  WARNING: No adversarial rows found in key!")

In [ ]:
import pandas as pd
import os

# File paths
per_case_csv = 'data/adversarial_judged_PER_CASE.csv'
summary_csv = 'data/adversarial_judged_SUMMARY.csv'

per_case_xlsx = 'data/adversarial_judged_PER_CASE.xlsx'
summary_xlsx = 'data/adversarial_judged_SUMMARY.xlsx'

# Load and convert
for src, dst in [(per_case_csv, per_case_xlsx), (summary_csv, summary_xlsx)]:
    if not os.path.exists(src):
        print(f"Warning: {src} not found. Skipping.")
        continue

    df = pd.read_csv(src)
    df.to_excel(dst, index=False)
    print(f"Converted {src} -> {dst} ({len(df)} rows)")

print("Done.")

### Development findings

In [ ]:
df = pd.read_csv('data/development_findings.csv')

new_findings = pd.DataFrame([
    {
        'case_id': 'DF-02',
        'persona': 'Maria',
        'recipe_title': 'Tempura',
        'claim_type': 'ingredients_dish',
        'set': 'validation',
        'judge_qwen': 'Supported',
        'judge_gpt-oss': 'Unsupported',
        'agreement': 'judges disagreed',
        'root_cause': 'The key-ingredient extractor (select_key_ingredients) selected "yolks" as the sole key ingredient for Tempura, an uninformative and misleading choice for a battered dish. The grounding fact itself was low quality, though not false.',
        'generator_behaviour': 'CORRECT (faithful). The generator reported only the supplied fact ("an appetizer featuring yolks"); it did not invent ingredients.',
        'judge_behaviour': 'gpt-oss flagged the awkward ingredient naming; qwen accepted it. The flag reflects extraction quality, not generator hallucination.',
        'finding': 'Ingredient extraction can surface uninformative ingredients for some recipes; a faithful explanation then inherits the awkwardness. This is a data-extraction limitation, not a grounding failure.',
        'action_taken': 'Recipe removed from the validation set and regenerated with a replacement from the same persona pool.',
        'status': 'Fixed - retained as extraction-quality example',
    },
    {
        'case_id': 'DF-03',
        'persona': 'James',
        'recipe_title': 'Steak and Potato Stir-fry',
        'claim_type': 'ingredients_dish',
        'set': 'validation',
        'judge_qwen': 'Supported',
        'judge_gpt-oss': 'Unsupported',
        'agreement': 'judges disagreed',
        'root_cause': 'The dish-type classifier (describe_dish_type) labelled a recipe titled "Stir-fry" as "stew", because DISH_TYPE_PRIORITY ranked the "stews" tag above "stir-fry" for a recipe carrying both tags.',
        'generator_behaviour': 'CORRECT (faithful). The generator reported the supplied dish type ("American stew"); it did not invent it.',
        'judge_behaviour': 'gpt-oss flagged the dish-type mismatch against the recipe title; qwen accepted it.',
        'finding': 'Dish-type classification from folksonomy tags can mislabel recipes carrying multiple course tags. A faithful explanation inherits the mislabel.',
        'action_taken': 'Recipe removed from the validation set and regenerated with a replacement from the same persona pool.',
        'status': 'Fixed - retained as extraction-quality example',
    },
])

df = pd.concat([df, new_findings], ignore_index=True)
df.to_csv('data/development_findings.csv', index=False)
print(f"Development findings now has {len(df)} cases:")
print(df[['case_id','recipe_title','status']].to_string(index=False))

### Third model for the judges' disagreements
Muse-Glimmer-30B-NVFP4, at temperature 0 like the two judges, decides the 17 claims on which they disagreed.

In [ ]:
import os, json
from openai import OpenAI

try:                                   
    from dotenv import load_dotenv
    load_dotenv(override=False)
except ImportError:
    pass

assert os.environ.get('LEHMUS_API_KEY_2'), \
    "LEHMUS_API_KEY_2 is not set in this kernel (restart the kernel after adding it to .env)"

# Same Lehmus gateway as the existing judges (LEHMUS_BASE_URL); only the key is new.
# If Lehmus gave the new key its own URL, put it in .env as LEHMUS_BASE_URL_2 and it is used instead.
LEHMUS_BASE_URL_2 = os.getenv('LEHMUS_BASE_URL_2') or os.getenv('LEHMUS_BASE_URL') or str(lehmus_client.base_url)
lehmus_client_2 = OpenAI(api_key=os.environ['LEHMUS_API_KEY_2'], base_url=LEHMUS_BASE_URL_2)
print('Gateway:', LEHMUS_BASE_URL_2)

THIRD_JUDGE_MODEL = 'Muse-Glimmer-30B-NVFP4'
THIRD_JUDGE_NAME  = 'Muse Glimmer 30B NVFP4'

try:
    available = [m.id for m in lehmus_client_2.models.list().data]
    print(f'Models this key can use ({len(available)}):', available)
    if THIRD_JUDGE_MODEL in available:
        print(f'OK: {THIRD_JUDGE_MODEL} is available with this key.')
    else:
        print(f'WARNING: {THIRD_JUDGE_MODEL} is not in the list. Check the exact model ID in the Lehmus portal '
              f'(it may be listed as "nvidia-muse-glimmer-30b-nvfp4") and set THIRD_JUDGE_MODEL to it.')
except Exception as e:                 
    print(f'Could not list models ({type(e).__name__}); the smoke test in Cell 2 will check the model ID.')


# ─────────────── CELL 2: run Muse Glimmer on the disputed claims, build the final label, compare with humans ───────────────

JUDGE_TEMPERATURE   = 0       # same as the Qwen / GPT-OSS judges 
JUDGE_TIMEOUT       = 600     # seconds per call
INCLUDE_ADVERSARIAL = True    # also judge the 8 adversarial disagreements
MAX_FAILS_IN_A_ROW  = 4

import re, time
import pandas as pd

DATA      = 'data'
KEY_PATH  = next(p for p in (f'{DATA}/annotation_KEY.csv', f'{DATA}/annotation_KEY_1.csv') if os.path.exists(p))
OUT_DS    = f'{DATA}/glimmer_tiebreak.csv'
OUT_KEY   = f'{DATA}/annotation_KEY_tiebreak.csv'
ANNOTATED = f'{DATA}/annotation_workbook_1_1.xlsx' if os.path.exists(f'{DATA}/annotation_workbook_1_1.xlsx') else None
VALID_V   = {'Supported', 'Unsupported'}
CLAIMS5   = ['ingredients_dish', 'goal_rank', 'healthiness', 'cuisine', 'dietary']

# ── 1) the disputed claims, from the annotation key
key = pd.read_csv(KEY_PATH).fillna('')
dis = key[key['judge_qwen'].isin(VALID_V) & key['judge_gpt-oss'].isin(VALID_V)
          & (key['judge_qwen'] != key['judge_gpt-oss'])].copy()
if not INCLUDE_ADVERSARIAL:
    dis = dis[dis['source'] == 'natural_disagree']
print(f"Disputed claims to send to Muse Glimmer: {len(dis)} -> {dis['source'].value_counts().to_dict()}")

# ── 2) rebuild the exact inputs the original judges saw
claims = pd.read_csv(f'{DATA}/claims_with_judges.csv').set_index('claim_id')
pool = pd.concat([pd.read_csv(f'{DATA}/explanations.csv'),
                  pd.read_csv(f'{DATA}/explanations_VALIDATION.csv')], ignore_index=True)
pool['explanation_id'] = (pool['persona_key'] + '__' + pool['recipe_title']).str[:80]
pool = pool.drop_duplicates('explanation_id').set_index('explanation_id')
adv = None
if (dis['source'] == 'adversarial').any():
    adv = pd.read_csv(f'{DATA}/adversarial_cases.csv')
    adv['claim_id'] = [f"adv__{r.persona_key}__{r.recipe_title}__{r.target_claim}"[:90] for r in adv.itertuples()]
    adv = adv.set_index('claim_id')

jobs = []
for _, k in dis.iterrows():
    cid = k['claim_id']
    if k['source'] == 'adversarial':
        r, ct = adv.loc[cid], adv.loc[cid]['target_claim']
    else:
        c = claims.loc[cid]
        r, ct = pool.loc[c['explanation_id']], c['claim_type']
    jobs.append(dict(row_id=int(k['row_id']), claim_id=cid, source=k['source'], claim_type=ct,
                     row=r, persona=TEST_PERSONAS[r['persona_key']]))


def last_json_object(raw):
    """Last top-level JSON object in the reply (skips reasoning text and stray braces)."""
    dec, pos, found = json.JSONDecoder(), 0, None
    while True:
        i = raw.find('{', pos)
        if i == -1:
            return found
        try:
            obj, end = dec.raw_decode(raw[i:])
            if isinstance(obj, dict) and obj:
                found = obj
            pos = i + end
        except json.JSONDecodeError:
            pos = i + 1


def normalise(v):
    if isinstance(v, dict):
        v = v.get('verdict', '')
    v = str(v).strip().strip('.').capitalize()
    return v if v in VALID_V else ''


def third_judge(row, persona, max_retries=3):
    """The thesis judge prompt, unchanged, sent to Muse Glimmer. Returns ({claim: verdict}, error)."""
    prompt, err = build_perclaim_judge_prompt(row, persona), ''
    for attempt in range(max_retries):
        try:
            resp = lehmus_client_2.chat.completions.create(
                model=THIRD_JUDGE_MODEL, messages=[{"role": "user", "content": prompt}],
                temperature=JUDGE_TEMPERATURE, timeout=JUDGE_TIMEOUT)
            raw = (resp.choices[0].message.content or '').strip()
            raw = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL)
            raw = re.sub(r'```json|```', '', raw)
            obj = last_json_object(raw)
            if obj:
                out = {k.lower(): normalise(v) for k, v in obj.items()}
                if any(out.get(c) for c in CLAIMS5):
                    return out, ''
            err = 'no usable JSON verdicts in reply'
        except Exception as e:
            err = f'{type(e).__name__}: {e}'
        time.sleep(3)
    return {}, err


# ── 3) resume from checkpoint, smoke-test one claim, then run the rest
done = {}
if os.path.exists(OUT_DS):
    for _, x in pd.read_csv(OUT_DS).fillna('').iterrows():
        if x['judge_glimmer'] in VALID_V:
            done[int(x['row_id'])] = x.to_dict()
    print(f"Resuming: {len(done)} claims already judged by Muse Glimmer")

pending = [j for j in jobs if j['row_id'] not in done]
test_out = None
if pending:
    t0 = time.time()
    test_out, test_err = third_judge(pending[0]['row'], pending[0]['persona'])
    print(f"Smoke test ({THIRD_JUDGE_MODEL}, {time.time() - t0:.0f}s):", test_out or test_err)
    assert test_out, f"Muse Glimmer did not return usable verdicts: {test_err}"

fails = 0
for n, j in enumerate(pending, start=1):
    out, err = (test_out, '') if j is pending[0] else third_judge(j['row'], j['persona'])
    verdict = out.get(j['claim_type'], '') or 'ERROR'
    done[j['row_id']] = dict(row_id=j['row_id'], claim_id=j['claim_id'], source=j['source'],
                             claim_type=j['claim_type'], judge_glimmer=verdict,
                             glimmer_model=THIRD_JUDGE_MODEL, error='' if verdict in VALID_V else err)
    pd.DataFrame(done.values()).to_csv(OUT_DS, index=False)
    fails = 0 if verdict in VALID_V else fails + 1
    print(f"  [{n}/{len(pending)}] Row {j['row_id']:>3} {j['claim_type']:<17} Muse Glimmer: {verdict} {err[:100]}")
    if fails >= MAX_FAILS_IN_A_ROW:
        raise RuntimeError(f"{fails} failures in a row (last: {err}). Fix and re-run this cell; it resumes.")
    time.sleep(0.4)

ds = pd.DataFrame(done.values())
missing = ds[~ds['judge_glimmer'].isin(VALID_V)]
if len(missing):
    print(f"\nWARNING: no Muse Glimmer verdict for Rows {missing['row_id'].tolist()}; re-run this cell to retry them.")

# ── 4) final LLM label for all 152 rows (no 'Disagreement' category) ──
#    natural_agreed   -> the two judges' shared verdict (unchanged)
#    natural_disagree -> majority of Qwen, GPT-OSS, Muse Glimmer (= Muse Glimmer's vote decides)
#    adversarial      -> the known label (unchanged; the planted error decides)
kt = key.merge(ds[['row_id', 'judge_glimmer']], on='row_id', how='left').fillna({'judge_glimmer': ''})
kt['llm_label_final'] = [(r['judge_glimmer'] if r['judge_glimmer'] in VALID_V else 'PENDING')
                         if r['source'] == 'natural_disagree' else r['known_label'] for _, r in kt.iterrows()]
kt['label_basis'] = kt['source'].map({'natural_agreed': 'Qwen + GPT-OSS agreed',
                                      'natural_disagree': 'majority of 3 LLMs (Muse Glimmer tie-break)',
                                      'adversarial': 'known label (planted error)'})
kt['glimmer_sided_with'] = [('Qwen' if r['judge_glimmer'] == r['judge_qwen'] else 'GPT-OSS')
                             if r['judge_glimmer'] in VALID_V else '' for _, r in kt.iterrows()]
kt.to_csv(OUT_KEY, index=False)
print(f"\nSaved {OUT_KEY}")

nd = kt[kt['source'] == 'natural_disagree']
print("Natural disagreements - Muse Glimmer sided with:", nd['glimmer_sided_with'].value_counts().to_dict())
if INCLUDE_ADVERSARIAL:
    ad = kt[(kt['source'] == 'adversarial') & kt['judge_glimmer'].isin(VALID_V)]
    print(f"Adversarial disagreements - Muse Glimmer caught the planted error: "
          f"{(ad['judge_glimmer'] == 'Unsupported').sum()}/{len(ad)}")

# ── 5) compare the final LLM label with the human majority
def kappa(a, b):
    a, b = pd.Series(list(a)), pd.Series(list(b))
    po = (a == b).mean()
    pe = sum((a == c).mean() * (b == c).mean() for c in set(a) | set(b))
    return (po - pe) / (1 - pe) if pe < 1 else float('nan')

if ANNOTATED:
    ann = pd.read_excel(ANNOTATED, header=6)
    ann = ann[pd.to_numeric(ann['Row'], errors='coerce').notna()].copy()
    ann['row_id'] = ann['Row'].astype(int)
    maj_col = next(c for c in ann.columns if str(c).startswith('Majority'))
    cmp = kt.merge(ann[['row_id', maj_col]].rename(columns={maj_col: 'human_majority'}), on='row_id')
    cmp = cmp[cmp['llm_label_final'].isin(VALID_V)]
    print("\nFinal LLM label vs human majority")
    for name, sub in (('All claims', cmp),
                      ('  the 17 tie-broken claims', cmp[cmp['source'] == 'natural_disagree']),
                      ('  the other 135 (supervisor: 80.7%, 0.606)', cmp[cmp['source'] != 'natural_disagree'])):
        print(f"  {name:<44} n={len(sub):>3}  agreement {(sub['llm_label_final'] == sub['human_majority']).mean():6.1%}"
              f"  kappa {kappa(sub['llm_label_final'], sub['human_majority']):.3f}")
    cmp.to_csv(f'{DATA}/annotation_KEY_tiebreak_vs_human.csv', index=False)

    print("\nThe disputed claims, by Row in the annotation workbook:")
    show = cmp[cmp['judge_glimmer'].isin(VALID_V)].sort_values('row_id')
    print(show[['row_id', 'source', 'judge_qwen', 'judge_gpt-oss', 'judge_glimmer',
                'glimmer_sided_with', 'llm_label_final', 'human_majority']].to_string(index=False))
else:
    print("\n(copy into data/ to get the comparison with the humans)")
